In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

from open_clip import create_model_and_transforms, get_tokenizer


In [2]:
device = "cuda"

model_name = "coca_ViT-L-14"
pretrained_path = "/project_antwerp/hbae/open_clip2/logs/finetune_HEG_final/checkpoints/epoch_5.pt"

model, preprocess_train, preprocess_val = create_model_and_transforms(
    model_name,
    pretrained=pretrained_path
)
tokenizer = get_tokenizer(model_name)
model = model.to(device)
model.eval()

print("Model loaded:", model_name)


Model loaded: coca_ViT-L-14


In [3]:
train_df = pd.read_csv("/project_antwerp/hbae/data/HEG_finetune_meta_gpulab.csv")
print(train_df.shape)
train_df.head()


(154763, 6)


,Unnamed: 0,label,sample_name,dataset,img_idx,img_path
0,GSM5494475_AAACACCAATAACTGC-1_hires,TTN IGKC MT-ND4 MT-CO1 MT-CO3 ACTA1 MT-ATP6 NE...,GSM5494475,GSE181300,GSM5494475_AAACACCAATAACTGC-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
1,GSM5494475_AAACAGCTTTCAGAAG-1_hires,IGKC IGHG4 IGLC2 IGHG1 IGHG3 COL1A2 COL1A1 THB...,GSM5494475,GSE181300,GSM5494475_AAACAGCTTTCAGAAG-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
2,GSM5494475_AAACAGGGTCTATATT-1_hires,IGKC IGHG4 IGHG3 IGLC2 B2M ACTB MALAT1 AQP1 IG...,GSM5494475,GSE181300,GSM5494475_AAACAGGGTCTATATT-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
3,GSM5494475_AAACATGGTGAGAGGA-1_hires,IGKC IGHG4 IGHG3 IGLC2 MALAT1 MT-CO3 TNNT3 MT-...,GSM5494475,GSE181300,GSM5494475_AAACATGGTGAGAGGA-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
4,GSM5494475_AAACCGGGTAGGTACC-1_hires,IGKC IGHG4 IGHG3 IGLC2 IGHG1 CD74 FTL IGLC1 MA...,GSM5494475,GSE181300,GSM5494475_AAACCGGGTAGGTACC-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...


In [4]:
def get_text_emb(sentences):
    all_emb = []

    for text in tqdm(sentences):
        tokens = tokenizer([text]).to(device)
        with torch.no_grad():
            emb = model.encode_text(tokens)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        all_emb.append(emb.cpu().numpy()[0])
    
    return np.vstack(all_emb)

train_sentences = train_df["label"].astype(str).tolist()
text_emb = get_text_emb(train_sentences)
print(text_emb.shape)

np.save("train_text_embedding.npy", text_emb)


100%|██████████| 154763/154763 [13:31<00:00, 190.77it/s]


(154763, 768)


In [5]:

from pathlib import Path
import h5py
import numpy as np
import torch
from tqdm import tqdm
from PIL import Image

def load_hdf5_patches(hdf5_path):
    """
    HDF5 파일 안에서 '이미지처럼 생긴(차원 3 or 4)' dataset 하나를 자동으로 찾아서 반환.
    """
    f = h5py.File(hdf5_path, "r")
    candidates = []

    def collect(name, obj):
        # dataset 이고, 차원이 3( H, W, C ) 또는 4( N, H, W, C / N, C, H, W ) 이상인 애들만 후보로
        if isinstance(obj, h5py.Dataset) and obj.ndim >= 3:
            candidates.append(name)

    f.visititems(collect)

    if not candidates:
        f.close()
        raise KeyError(f"No image-like dataset (ndim>=3) found in HDF5 file: {hdf5_path}")

    # 일단 첫 번째 후보를 사용 (필요하면 출력해서 확인 가능)
    key = candidates[0]
    print(f"▶ Using dataset '{key}' from {hdf5_path}")
    imgs = f[key]     # shape: (N, H, W, C) or (N, C, H, W) or 비슷한 형태

    return f, imgs

from pathlib import Path

# 슬라이드 patch 폴더들이 모여 있는 루트
tcga_root = Path("/project_antwerp/TCGA-HNSC/TCGA_patch/")

# 각 슬라이드 폴더 안의 모든 .h5 / .hdf5 파일 수집
h5_list = sorted(list(tcga_root.rglob("*.hdf5")))

print("HDF5 파일 개수:", len(h5_list))
for p in h5_list[:5]:
    print(p)






HDF5 파일 개수: 528
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1.377dc7cb-f029-4bf5-bdd5-8997f91e551a/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1.604893b7-5fcb-4bab-b2cd-6cad1c8ab456/TCGA-BA-4077-01Z-00-DX1.hdf5


In [6]:
from pathlib import Path

tcga_root = Path("/project_antwerp/TCGA-HNSC/TCGA_patch")

h5_list = sorted(list(tcga_root.rglob("*.h5")) + list(tcga_root.rglob("*.hdf5")))
print("총 HDF5:", len(h5_list))

# slide_id = 파일 이름(확장자 제거)
slide_to_path = {}

for p in h5_list:
    slide_id = p.stem  # 'TCGA-BA-4074-01Z-00-DX1'
    cur = slide_to_path.get(slide_id)

    if cur is None:
        slide_to_path[slide_id] = p
    else:
        # 보통 UUID 안 붙은 “짧은 경로”를 우선으로 사용
        if len(str(p)) < len(str(cur)):
            slide_to_path[slide_id] = p

print("슬라이드당 하나로 dedup 후 개수:", len(slide_to_path))
dedup_h5_list = list(slide_to_path.values())[:5]
for p in dedup_h5_list:
    print(p)


총 HDF5: 528
슬라이드당 하나로 dedup 후 개수: 331
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5151-01Z-00-DX1/TCGA-BA-5151-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5153-01Z-00-DX1/TCGA-BA-5153-01Z-00-DX1.hdf5


In [7]:
import pandas as pd
import numpy as np

tcga_rna_csv = "/project_antwerp/TCGA-HNSC/ref_file.csv"  # 너가 실제 쓰는 경로로
rna_df = pd.read_csv(tcga_rna_csv)

# slide_id 정리
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].str.split('.').str[0]
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df = rna_df.set_index(first_col)

# 숫자 컬럼만 남기고 정리 (전에 하던 대로)
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

rna_df = rna_df.select_dtypes(include=[np.number])

print("RNA shape:", rna_df.shape)
print("슬라이드 수:", len(rna_df))


RNA shape: (461, 24778)
슬라이드 수: 461


In [8]:
import pandas as pd

# 예: rna_df.index 가 slide_id 들 (461개)
rna_slide_ids = set(rna_df.index.astype(str))

# dedup된 HDF5 중에서 RNA가 있는 슬라이드만 선택
matched_h5_paths = [p for sid, p in slide_to_path.items() if sid in rna_slide_ids]

print("RNA와 매칭된 슬라이드 수:", len(matched_h5_paths))
for p in matched_h5_paths[:5]:
    print(p)


RNA와 매칭된 슬라이드 수: 331
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5151-01Z-00-DX1/TCGA-BA-5151-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5153-01Z-00-DX1/TCGA-BA-5153-01Z-00-DX1.hdf5


In [9]:
import h5py
import numpy as np

class H5ImageList:
    """여러 3D dataset(각각이 한 patch)을 list처럼 다루기 위한 래퍼"""
    def __init__(self, h5file, keys):
        self.h5file = h5file
        self.keys = keys  # 패치 이름 리스트

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        # h5py Dataset -> numpy array로 로드
        return self.h5file[self.keys[idx]][:]

def load_hdf5_patches(hdf5_path):
    f = h5py.File(hdf5_path, "r")

    # 1) 먼저 4D batched dataset이 있는지 확인 (N, H, W, C) or (N, C, H, W)
    for k in f.keys():
        ds = f[k]
        if isinstance(ds, h5py.Dataset) and ds.ndim == 4:
            # 기존 코드와 동일하게 동작하도록 그대로 반환
            return f, ds

    # 2) 아니면, 여러 개의 3D (H, W, C) patch dataset으로 이루어진 케이스 처리
    img_keys = [
        k for k in f.keys()
        if isinstance(f[k], h5py.Dataset) and f[k].ndim == 3
    ]

    if img_keys:
        # 순서 고정하고 싶으면 정렬
        img_keys = sorted(img_keys)  # 이름으로 정렬 (예: "89600_50176", ...)
        imgs = H5ImageList(f, img_keys)
        return f, imgs

    # 3) 아무것도 못 찾으면 에러
    f.close()
    raise KeyError(f"No image dataset found in HDF5 file: {hdf5_path}")


In [10]:
from PIL import Image
from tqdm import tqdm
import torch
import numpy as np  # ← 이거!

def extract_embeddings_from_hdf5(model, preprocess_val, hdf5_path, device):
    f, imgs = load_hdf5_patches(hdf5_path)
    emb_list = []

    print(f"Found {len(imgs)} patches in {hdf5_path}")

    for i in tqdm(range(len(imgs))):
        arr = imgs[i]  # numpy array (H, W, C) or (C, H, W) 또는 래퍼에서 가져온 것

        # If array is CHW, convert to HWC
        if arr.ndim == 3 and arr.shape[0] == 3:
            arr = np.transpose(arr, (1, 2, 0))

        img = Image.fromarray(arr.astype("uint8"))
        img = preprocess_val(img).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = model.encode_image(img)
            emb = emb / emb.norm(dim=1, keepdim=True)

        emb_list.append(emb.cpu().numpy()[0])

    f.close()
    return np.vstack(emb_list)
def extract_embeddings_from_hdf5_batched(model, preprocess_val, hdf5_path, device, batch_size=64):
    f, imgs = load_hdf5_patches(hdf5_path)
    print(f"Found {len(imgs)} patches in {hdf5_path}")

    emb_out = []
    batch_tensors = []

    for i in tqdm(range(len(imgs))):
        arr = imgs[i]
        if arr.ndim == 3 and arr.shape[0] == 3:
            arr = np.transpose(arr, (1, 2, 0))

        img = Image.fromarray(arr.astype("uint8"))
        batch_tensors.append(preprocess_val(img))

        if len(batch_tensors) == batch_size or i == len(imgs) - 1:
            x = torch.stack(batch_tensors, dim=0).to(device)
            with torch.no_grad():
                emb = model.encode_image(x)
                emb = emb / emb.norm(dim=1, keepdim=True)
            emb_out.append(emb.cpu().numpy())
            batch_tensors = []

    f.close()
    return np.vstack(emb_out)



In [11]:
from PIL import Image
from tqdm import tqdm
import torch

from pathlib import Path

save_dir = Path("./tcga_embeddings")
save_dir.mkdir(exist_ok=True)

all_embeddings = {}

for h5 in matched_h5_paths:
    slide_id = h5.stem.split(".")[0]
    print(f"\nProcessing slide: {slide_id}")

    try:
        emb = extract_embeddings_from_hdf5_batched(model, preprocess_val, h5, device)
    except KeyError as e:
        print("❌ Skipping (no image dataset):", h5)
        continue

    np.save(save_dir / f"{slide_id}.npy", emb)
    all_embeddings[slide_id] = emb

print("✅ 전체 슬라이드 embedding 완료")

# TCGA HDF5 파일에서 이미지 임베딩을 추출해 저장합니다.



Processing slide: TCGA-4P-AA8J-01Z-00-DX1
Found 11834 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5


100%|██████████| 11834/11834 [02:48<00:00, 70.11it/s] 



Processing slide: TCGA-BA-4074-01Z-00-DX1
Found 1033 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5


100%|██████████| 1033/1033 [00:14<00:00, 72.16it/s]



Processing slide: TCGA-BA-4077-01Z-00-DX1
Found 1749 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5


...


Processing slide: TCGA-WA-A7H4-01Z-00-DX1
Found 7152 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-WA-A7H4-01Z-00-DX1/TCGA-WA-A7H4-01Z-00-DX1.hdf5


100%|██████████| 7152/7152 [01:37<00:00, 73.22it/s] 



Processing slide: TCGA-WA-A7H4-01Z-00-DX2
Found 1948 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-WA-A7H4-01Z-00-DX2/TCGA-WA-A7H4-01Z-00-DX2.hdf5


100%|██████████| 1948/1948 [00:26<00:00, 72.72it/s] 

✅ 전체 슬라이드 embedding 완료


In [ ]:
import os
import numpy as np
import h5py
from PIL import Image
from tqdm import tqdm
import torch
from pathlib import Path
from contextlib import nullcontext


def load_hdf5_patches(hdf5_path):
    """
    Returns:
      f: h5py.File
      imgs: either
        - h5py.Dataset (shape: (N,H,W,C) or (N,C,H,W))
        - list[str] keys where each key is a patch dataset (shape: (H,W,C))
    """
    f = h5py.File(hdf5_path, "r")

    # Case A: common single-dataset layout
    candidate_keys = ["imgs", "images", "img", "patches", "data", "x"]
    for k in candidate_keys:
        if k in f and isinstance(f[k], h5py.Dataset):
            return f, f[k]

    # Case B: root has many datasets per patch (your case: "89600_50176", ...)
    keys = []
    for k in f.keys():
        obj = f[k]
        if isinstance(obj, h5py.Dataset):
            shp = obj.shape
            # typically (256,256,3) or (H,W,3)
            if len(shp) == 3 and shp[-1] in (1, 3, 4):
                keys.append(k)

    if len(keys) == 0:
        f.close()
        raise KeyError(f"No image dataset found in HDF5 file: {hdf5_path}")

    keys.sort()
    return f, keys


def extract_embeddings_from_hdf5(
    model,
    preprocess_val,
    hdf5_path,
    device,
    batch_size=64,
    use_fp16=True,
    verbose_skip=False,
):
    device = torch.device(device) if isinstance(device, str) else device
    f, imgs = load_hdf5_patches(hdf5_path)

    is_dataset = isinstance(imgs, h5py.Dataset)
    n = imgs.shape[0] if is_dataset else len(imgs)

    print(f"Found {n} patches in {hdf5_path}")

    # FP16 only on CUDA
    autocast_enabled = (use_fp16 and device.type == "cuda")
    autocast_ctx = torch.cuda.amp.autocast if autocast_enabled else nullcontext

    def _get_patch(i):
        if is_dataset:
            return imgs[i]          # returns numpy-like array
        else:
            return f[imgs[i]][()]   # dataset -> numpy array

    emb_chunks = []
    batch_tensors = []
    skipped = 0

    model.eval()

    for i in tqdm(range(n)):
        try:
            arr = _get_patch(i)

            # if CHW -> HWC
            if arr.ndim == 3 and arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):
                arr = np.transpose(arr, (1, 2, 0))

            # ensure uint8
            if arr.dtype != np.uint8:
                arr = arr.astype(np.uint8)

            img = Image.fromarray(arr)
            batch_tensors.append(preprocess_val(img))

        except Exception as e:
            skipped += 1
            if verbose_skip:
                print(f"[skip] patch {i}: {e}")
            continue

        # flush batch
        if len(batch_tensors) == batch_size or i == n - 1:
            x = torch.stack(batch_tensors, dim=0).to(device, non_blocking=True)
            batch_tensors = []

            with torch.no_grad():
                with autocast_ctx():
                    emb = model.encode_image(x)

                emb = emb / emb.norm(dim=1, keepdim=True)
                emb_chunks.append(emb.float().cpu().numpy())

    f.close()

    if skipped > 0:
        print(f"⚠️ Skipped patches due to errors: {skipped}/{n}")

    if len(emb_chunks) == 0:
        # return empty (safer than crash)
        return np.zeros((0, 0), dtype=np.float32)

    return np.vstack(emb_chunks)


def atomic_save_npy(out_path: Path, arr: np.ndarray):
    """
    Safe save: write to temp then atomic replace.
    Fixes: np.save auto-appending .npy if given a string path.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")  # e.g. file.npy.tmp

    # IMPORTANT: use file handle so np.save doesn't append ".npy"
    with open(tmp_path, "wb") as ftmp:
        np.save(ftmp, arr)

    os.replace(tmp_path, out_path)  # atomic on POSIX


# =========================
# Main loop (resume-capable)
# =========================

save_dir = Path("./tcga_embeddings")
save_dir.mkdir(exist_ok=True)

all_embeddings = {}

for h5 in matched_h5_paths:
    h5 = Path(h5)
    slide_id = h5.stem.split(".")[0]
    out_path = save_dir / f"{slide_id}.npy"

    # resume: already exists -> skip
    if out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        try:
            all_embeddings[slide_id] = np.load(out_path, mmap_mode="r")
        except Exception:
            pass
        continue

    print(f"\nProcessing slide: {slide_id}")

    try:
        emb = extract_embeddings_from_hdf5(
            model=model,
            preprocess_val=preprocess_val,
            hdf5_path=h5,
            device=device,
            batch_size=64,     # tune: 32/64/128
            use_fp16=True,     # fp16 only if cuda
            verbose_skip=False
        )

        atomic_save_npy(out_path, emb)
        all_embeddings[slide_id] = emb
        print(f"✅ Saved: {out_path} | shape={emb.shape}")

    except KeyError as e:
        print(f"❌ Skipping (no image dataset): {h5} | {e}")
        continue
    except Exception as e:
        print(f"❌ Error on {slide_id}: {e}")
        continue

print("✅ 전체 슬라이드 embedding 완료")


## Top 300 Genes 리스트 생성

논문과 동일하게 top 300 expressed genes를 선택합니다.
`all_shared_genes.txt`의 순서와 `combined_expression_matrix.npy`의 컬럼 순서가 1:1로 대응됩니다.


In [13]:
import numpy as np
from pathlib import Path

# =========================
# Top 300 Genes 생성 (방법 A: 평균 발현 기준)
# =========================

# 유전자 목록 로드
genes_all = [line.strip() for line in open("/project_antwerp/hbae/data/all_shared_genes.txt") if line.strip()]
print(f"✅ 전체 유전자 수: {len(genes_all)}")

# Expression matrix 로드
X = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy")  # (Ntrain, 12009)
print(f"✅ Expression matrix shape: {X.shape}")

# 평균 발현이 높은 300개 선택
gene_mean = X.mean(axis=0)  # (12009,)
topk = 300
top_idx = np.argsort(-gene_mean)[:topk]  # 내림차순 정렬 후 상위 300개

top_genes = [genes_all[i] for i in top_idx]

# 저장
with open("top300_genes.txt", "w") as f:
    f.write("\n".join(top_genes))

print(f"✅ Top 300 genes saved: top300_genes.txt")
print(f"   예시: {top_genes[:10]}")

# 인덱스 매핑 생성 (나중에 사용)
gene2idx = {g: i for i, g in enumerate(genes_all)}
top_idx_array = np.array([gene2idx[g] for g in top_genes], dtype=np.int64)

print(f"✅ Top 300 인덱스 준비 완료: shape={top_idx_array.shape}")


✅ 전체 유전자 수: 12009
✅ Expression matrix shape: (154763, 12009)
✅ Top 300 genes saved: top300_genes.txt
   예시: ['S100A8', 'KRT14', 'S100A9', 'S100A2', 'B2M', 'ACTB', 'KRT5', 'S100A11', 'FABP5', 'KRT16']
✅ Top 300 인덱스 준비 완료: shape=(300,)


## 패치 레벨 예측 (Loki 방식 + Top 300 Genes + ReLU)

기존 Cell 15 코드를 수정하여:
1. **Top 300 genes만** 저장
2. **ReLU 적용**: 음수 similarity 제거로 예측 안정성 향상
3. 논문의 Equation (16) 방식 유지


In [14]:
import numpy as np
from pathlib import Path
import sys

# Loki 모듈 import
loki_path = "/project_antwerp/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        """Loki PredEx 함수 직접 구현"""
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression_loki_top300")  # Top 300 genes 결과 저장
OUT_DIR.mkdir(exist_ok=True)

EPS = 1e-8
PATCH_CHUNK = 256    # 메모리 효율을 위한 청크 크기

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize (cosine similarity = dot product)
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

print(f"✅ 데이터 로드 완료: {Ntrain}개 학습 샘플, {G}개 유전자")

# =========================
# Top 300 genes 인덱스 로드 및 train_expr 필터링
# =========================
genes_all = [line.strip() for line in open("/project_antwerp/hbae/data/all_shared_genes.txt") if line.strip()]
top_genes = [line.strip() for line in open("/project_antwerp/hbae/script/top300_genes.txt") if line.strip()]

gene2idx = {g: i for i, g in enumerate(genes_all)}
top_idx = np.array([gene2idx[g] for g in top_genes], dtype=np.int64)

# 🚀 성능 최적화: train_expr를 미리 Top 300개만 필터링 (12,009 → 300)
train_expr_top300 = train_expr[:, top_idx]  # (Ntrain, 300)

print(f"✅ Top 300 genes 인덱스 로드: {len(top_idx)}개")
print(f"✅ train_expr 필터링 완료: {train_expr.shape} → {train_expr_top300.shape}")

# =========================
# Slide loop - Loki 방식 (Top 300 + ReLU)
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    # 🔄 Resume 기능: 기존 파일이 있으면 건너뛰기
    FORCE_RECOMPUTE = False  # 기본값: 기존 결과 사용
    
    if not FORCE_RECOMPUTE and out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id} (Loki 방식 + Top 300 + ReLU)")

    # 이미지 임베딩 로드 및 정규화
    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    # 패치별로 청크 단위 처리 (메모리 효율)
    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        img_chunk = img_emb[p0:p1]  # (chunk_size, 768)
        
        # 🔑 Loki 방식: 모든 similarity 계산
        similarity = img_chunk @ train_text_emb.T  # (chunk_size, Ntrain)
        
        # 🔑 ReLU 적용: 음수 similarity 제거 (예측 안정성 향상)
        similarity = np.maximum(similarity, 0.0)
        
        # 🔑 Loki PredEx 함수 사용 (Top 300 genes만 계산 - 빠름!)
        pred_chunk = loki.predex.predict_st_gene_expr(similarity, train_expr_top300)  # (chunk_size, 300)
        pred_chunks.append(pred_chunk)

    pred_expr = np.vstack(pred_chunks)  # (P, 300)
    
    # 저장
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"
    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape} (Top 300 genes)")

print("\n✅ Loki 방식 (Top 300 + ReLU)으로 전체 슬라이드 예측 완료")
print(f"📁 결과 저장 위치: {OUT_DIR}")
print(f"📄 Gene 순서 파일: top300_genes.txt")


✅ Loki 모듈 로드 성공
Train text: (154763, 768)
Train expr: (154763, 12009)
✅ 데이터 로드 완료: 154763개 학습 샘플, 12009개 유전자
✅ Top 300 genes 인덱스 로드: 300개
✅ train_expr 필터링 완료: (154763, 12009) → (154763, 300)

🔹 Processing slide: TCGA-4P-AA8J-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (11834, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-4074-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (1033, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-4077-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (1749, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-5151-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (1444, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-5153-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (2777, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-6870-01Z-00-DX1 (Loki 방식 + Top 300 + ReLU)
✅ Saved predicted expression: (157, 300) (Top 300 genes)

🔹 Processing slide: TCGA-BA-

In [15]:
## 슬라이드 레벨 예측 (Top 300 Genes)

패치 레벨 예측값들을 평균내서 슬라이드 레벨로 집계합니다.


SyntaxError: invalid syntax (2938006200.py, line 3)

이 코드는 TCGA HDF5 파일에서 이미지 패치를 읽어 vision 모델로 임베딩을 추출합니다.

## 전체 구조

### 1. `load_hdf5_patches` 함수 (11-42줄)
HDF5 파일에서 이미지 데이터를 찾아 반환합니다.

- Case A: 단일 데이터셋 구조
  - `"imgs"`, `"images"`, `"patches"` 등 일반적인 키를 확인
  - 찾으면 해당 데이터셋 반환
- Case B: 패치별 개별 데이터셋 구조
  - 루트에 여러 데이터셋이 있는 경우 (예: `"89600_50176"`, `"89600_50177"` 등)
  - 각 키가 3차원 배열(H, W, C)인지 확인해 패치 키 목록 반환
  - 이미지 데이터가 없으면 `KeyError` 발생

### 2. `extract_embeddings_from_hdf5` 함수 (45-120줄)
HDF5에서 이미지 패치를 읽어 임베딩을 추출합니다.

- 초기 설정
  - HDF5 파일 열기, 패치 개수 확인
  - FP16 사용 여부 결정 (CUDA에서만)
- 배치 처리
  - 각 패치를 순회하며:
    - 패치 배열 읽기 (Case A/B 자동 처리)
    - CHW → HWC 변환 (필요 시)
    - uint8로 변환 후 PIL Image로 변환
    - 전처리 후 배치에 추가
  - 배치가 차면:
    - GPU로 전송
    - 모델의 `encode_image`로 임베딩 추출
    - L2 정규화 후 CPU로 이동해 저장
- 에러 처리
  - 개별 패치 오류는 건너뛰고 계속 진행
  - 건너뛴 패치 수 출력

### 3. `atomic_save_npy` 함수 (123-137줄)
안전하게 numpy 배열을 저장합니다.

- 임시 파일에 저장 후 원자적 교체
- 디렉토리 자동 생성
- 중간 실패 시 기존 파일 유지

### 4. 메인 루프 (143-187줄)
모든 HDF5 파일을 처리합니다.

- 각 HDF5 파일에 대해:
  1. 슬라이드 ID 추출
  2. 출력 파일 경로 확인
  3. 이미 존재하면 건너뛰기 (재개 가능)
  4. 없으면 임베딩 추출 후 저장
  5. 메모리 딕셔너리에 로드
- 에러 처리
  - `KeyError`: 이미지 데이터 없음 → 건너뛰기
  - 기타 예외: 에러 메시지 출력 후 다음 파일로 진행

## 주요 특징

1. 두 가지 HDF5 구조 지원 (단일 데이터셋 / 패치별 데이터셋)
2. 배치 처리로 GPU 효율 향상
3. FP16 사용으로 메모리/속도 개선 (CUDA에서)
4. 재개 가능: 이미 처리된 파일은 건너뜀
5. 안전한 저장: 임시 파일 사용으로 중간 실패 방지
6. 에러 복원력: 개별 패치/파일 오류 시에도 계속 진행

## 결과

- 각 슬라이드별로 `{slide_id}.npy` 파일 생성
- `all_embeddings` 딕셔너리에 메모리 로드
- 각 파일은 `(num_patches, embedding_dim)` 형태 (예: `(11834, 768)`)

이 임베딩은 이후 유전자 발현 예측에 사용됩니다.

맞습니다. 이 코드는 **TCGA HDF5 파일에 저장된 이미지 패치들에서 vision 모델을 사용해 임베딩을 추출**합니다.

## 정확한 요약

1. 입력: TCGA 슬라이드의 HDF5 파일
   - 각 파일에는 WSI에서 추출한 이미지 패치들이 저장되어 있음
   - 예: `TCGA-XX-XXXX-01Z-00-DX1.h5` → 수천~수만 개의 패치 이미지

2. 처리: 각 패치를 vision 모델에 통과
   - CoCa 모델의 `encode_image`로 임베딩 벡터 생성
   - 각 패치 → 768차원 벡터

3. 출력: 슬라이드별 임베딩 파일
   - `{slide_id}.npy`: `(num_patches, 768)` 형태
   - 예: `(11834, 768)` = 11,834개 패치, 각각 768차원 임베딩

## 전체 파이프라인에서의 역할

```
TCGA WSI 이미지 
  ↓ (패치 추출)
HDF5 파일들 (이미지 패치 저장)
  ↓ [이 코드]
이미지 임베딩 (.npy 파일들)
  ↓ (다음 단계)
유전자 발현 예측
```

이 임베딩은 이후 텍스트 임베딩(유전자 이름)과의 유사도 계산에 사용되어 유전자 발현을 예측합니다.

In [2]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# =========================
# Config
# =========================
PATCH_PRED_DIR = Path("tcga_predicted_expression_loki_top300")  # 패치 레벨 예측 결과
SLIDE_PRED_DIR = Path("tcga_pred_expr_loki_top300")  # 슬라이드 레벨 예측 결과
SLIDE_PRED_DIR.mkdir(exist_ok=True)

# =========================
# 슬라이드 레벨 집계
# =========================
print("="*60)
print("슬라이드 레벨 예측 집계 (Top 300 Genes)")
print("="*60)

for pred_file in tqdm(sorted(PATCH_PRED_DIR.glob("*.npy"))):
    if pred_file.name.endswith(".npy.tmp"):
        continue
    
    slide_id = pred_file.stem
    out_path = SLIDE_PRED_DIR / f"{slide_id}.npy"
    
    # Resume: 이미 있으면 건너뛰기
    if out_path.exists():
        continue
    
    # 패치 레벨 예측 로드
    patch_pred = np.load(pred_file)  # (num_patches, 300)
    
    # 슬라이드 레벨로 평균 (패치별 평균)
    slide_pred = patch_pred.mean(axis=0)  # (300,)
    
    # 저장
    np.save(out_path, slide_pred)

print(f"\n✅ 슬라이드 레벨 예측 완료")
print(f"📁 결과 저장 위치: {SLIDE_PRED_DIR}")
print(f"📊 Shape: (num_slides, 300)")


슬라이드 레벨 예측 집계 (Top 300 Genes)


100%|██████████| 331/331 [00:00<00:00, 4310.92it/s]


✅ 슬라이드 레벨 예측 완료
📁 결과 저장 위치: tcga_pred_expr_loki_top300
📊 Shape: (num_slides, 300)


In [ ]:
## TCGA Bulk RNA-seq 평가 (Top 300 Genes)

TCGA bulk RNA-seq 데이터와 비교하여 PCC와 MAE를 계산합니다.


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
from tqdm import tqdm

# =========================
# 경로 설정
# =========================
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_DIR = Path("tcga_pred_expr_loki_top300")  # 슬라이드 레벨 예측 결과
TOP300_GENES_FILE = "top300_genes.txt"

# =========================
# 정규화 함수
# =========================
def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

# =========================
# 1. TCGA RNA-seq 데이터 로드
# =========================
print("="*60)
print("1. TCGA RNA-seq 데이터 로드")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

# wsi_file_name 컬럼에서 slide_id 추출
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]  # 경로에서 파일명만 추출
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)  # 정규화
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    # 첫 번째 컬럼이 slide_id인 경우
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])
if 'wsi_file_name' in rna_df.columns:
    rna_df = rna_df.drop(columns=['wsi_file_name'])

# 숫자 컬럼만 선택 (gene expression 컬럼)
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# rna_ prefix 제거 및 정규화
rna_gene_names = [c.replace("rna_", "") for c in rna_expr_df.columns]
rna_gene_norm = [norm_gene(g) for g in rna_gene_names]

# RNA gene map
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map (이미 정규화된 slide_id가 index)
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")
print(f"RNA slide 예시: {list(rna_slide_map.keys())[:5]}")

# =========================
# 2. Top 300 genes 로드
# =========================
print("\n" + "="*60)
print("2. Top 300 genes 로드")
print("="*60)

top_genes = [line.strip() for line in open(TOP300_GENES_FILE) if line.strip()]
top_genes_norm = [norm_gene(g) for g in top_genes]

print(f"Top 300 genes: {len(top_genes)}")

# =========================
# 3. 공통 genes 찾기
# =========================
print("\n" + "="*60)
print("3. 공통 genes 찾기")
print("="*60)

common_genes = sorted(set(top_genes_norm) & set(rna_gene_map.keys()))

if len(common_genes) < len(top_genes_norm):
    print(f"⚠️ 경고: Top 300 중 {len(top_genes_norm) - len(common_genes)}개가 RNA-seq에 없음")
    print(f"   최종 비교 genes: {len(common_genes)}개")

# 인덱스 매핑
rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([top_genes_norm.index(g) for g in common_genes], dtype=np.int64)

print(f"✅ 공통 genes: {len(common_genes)}개")

# =========================
# 4. 공통 slides 찾기
# =========================
print("\n" + "="*60)
print("4. 공통 slides 찾기")
print("="*60)

pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids_raw = [p.stem for p in pred_files]
pred_sids = [norm_sid(sid) for sid in pred_sids_raw]
pred_sid_set = set(pred_sids)

print(f"Pred slides: {len(pred_sid_set)}")
print(f"Pred slide 예시: {list(pred_sid_set)[:5]}")

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))

print(f"Common slides: {len(common_sids)}")

# 디버깅: 매칭 안 되는 경우 샘플 출력
if len(common_sids) == 0:
    print("\n⚠️ 디버깅 정보:")
    print(f"   Pred slides (first 10): {list(pred_sid_set)[:10]}")
    print(f"   RNA slides (first 10): {list(rna_slide_map.keys())[:10]}")
    print(f"   Pred slides not in RNA: {list(pred_sid_set - set(rna_slide_map.keys()))[:10]}")
    print(f"   RNA slides not in Pred: {list(set(rna_slide_map.keys()) - pred_sid_set)[:10]}")
    raise RuntimeError("공통 slide가 0입니다. 위 디버깅 정보를 확인하세요.")

# =========================
# 5. 예측값과 실제값 로드
# =========================
print("\n" + "="*60)
print("5. 예측값과 실제값 로드")
print("="*60)

pred_list = []
rna_list = []
sid_list = []

for sid in tqdm(common_sids):
    # 예측값 로드 (원본 파일명 찾기)
    pred_file = None
    for p in pred_files:
        if norm_sid(p.stem) == sid:
            pred_file = p
            break
    
    if pred_file is None or not pred_file.exists():
        continue
    
    pred = np.load(pred_file)  # (300,)
    pred_selected = pred[pred_idx]  # 공통 genes만 선택
    
    # 실제값 로드
    rna_idx_slide = rna_slide_map[sid]
    rna = rna_expr_df.iloc[rna_idx_slide, rna_idx].values  # 공통 genes만 선택
    
    pred_list.append(pred_selected)
    rna_list.append(rna)
    sid_list.append(sid)

pred_array = np.array(pred_list)  # (num_slides, num_common_genes)
rna_array = np.array(rna_list)   # (num_slides, num_common_genes)

print(f"✅ 로드 완료: {pred_array.shape}")

# =========================
# 6. 평가 메트릭 계산
# =========================
print("\n" + "="*60)
print("6. 평가 메트릭 계산")
print("="*60)

# Overall PCC (flatten)
pred_flat = pred_array.flatten()
rna_flat = rna_array.flatten()
overall_pcc, _ = pearsonr(pred_flat, rna_flat)

# Gene-wise PCC
gene_pccs = []
gene_maes = []

for i in range(len(common_genes)):
    gene_pred = pred_array[:, i]
    gene_rna = rna_array[:, i]
    
    # PCC
    pcc, _ = pearsonr(gene_pred, gene_rna)
    if not np.isnan(pcc):
        gene_pccs.append(pcc)
    
    # MAE
    mae = np.mean(np.abs(gene_pred - gene_rna))
    gene_maes.append(mae)

gene_pccs = np.array(gene_pccs)
gene_maes = np.array(gene_maes)

print(f"📌 Overall PCC (flatten): {overall_pcc:.4f}")
print(f"📌 Gene-wise PCC mean: {gene_pccs.mean():.4f}")
print(f"📌 Gene-wise PCC std: {gene_pccs.std():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_maes.mean():.4f}")

# Top/Bottom PCC genes
gene_pcc_dict = {common_genes[i]: gene_pccs[i] for i in range(len(common_genes)) if not np.isnan(gene_pccs[i])}
sorted_genes = sorted(gene_pcc_dict.items(), key=lambda x: x[1], reverse=True)

print(f"\n✅ Top 10 PCC genes:")
for gene, pcc in sorted_genes[:10]:
    print(f"   {gene}: PCC={pcc:.4f}")

print(f"\n❌ Bottom 10 PCC genes:")
for gene, pcc in sorted_genes[-10:]:
    print(f"   {gene}: PCC={pcc:.4f}")

# =========================
# 7. 결과 저장
# =========================
print("\n" + "="*60)
print("7. 결과 저장")
print("="*60)

np.save("eval_loki_top300_common_sids.npy", np.array(common_sids))
np.save("eval_loki_top300_common_genes.npy", np.array(common_genes))
np.save("eval_loki_top300_gene_pcc.npy", gene_pccs)
np.save("eval_loki_top300_gene_mae.npy", gene_maes)

print("✅ 저장 완료:")
print("   - eval_loki_top300_common_sids.npy")
print("   - eval_loki_top300_common_genes.npy")
print("   - eval_loki_top300_gene_pcc.npy")
print("   - eval_loki_top300_gene_mae.npy")

print("\n" + "="*60)
print("✅ Loki Top 300 평가 완료!")
print("="*60)


1. TCGA RNA-seq 데이터 로드
RNA slides: 461, RNA genes: 24778
RNA slide 예시: ['TCGA-CV-6950-01Z-00-DX1', 'TCGA-D6-6515-01Z-00-DX1', 'TCGA-D6-6515-01Z-00-DX2', 'TCGA-F7-7848-01Z-00-DX1', 'TCGA-IQ-A61I-01Z-00-DX1']

2. Top 300 genes 로드
Top 300 genes: 300

3. 공통 genes 찾기
⚠️ 경고: Top 300 중 10개가 RNA-seq에 없음
   최종 비교 genes: 290개
✅ 공통 genes: 290개

4. 공통 slides 찾기
Pred slides: 331
Pred slide 예시: ['TCGA-P3-A6T6-01Z-00-DX1', 'TCGA-CV-6433-01Z-00-DX1', 'TCGA-CV-7423-01Z-00-DX1', 'TCGA-UF-A7JK-01Z-00-DX1', 'TCGA-CV-7410-01Z-00-DX1']
Common slides: 331

5. 예측값과 실제값 로드


100%|██████████| 331/331 [00:00<00:00, 869.23it/s]


✅ 로드 완료: (331, 290)

6. 평가 메트릭 계산
📌 Overall PCC (flatten): 0.5489
📌 Gene-wise PCC mean: -0.0017
📌 Gene-wise PCC std: 0.0929
📌 Gene-wise MAE mean: 3.1348

✅ Top 10 PCC genes:
   GJA1: PCC=0.2202
   DSTN: PCC=0.2119
   KRT15: PCC=0.1960
   CD44: PCC=0.1894
   ANXA2: PCC=0.1858
   HIF1A: PCC=0.1738
   PGAM1: PCC=0.1669
   KRT6B: PCC=0.1612
   AHNAK2: PCC=0.1604
   TUBB6: PCC=0.1568

❌ Bottom 10 PCC genes:
   MSN: PCC=-0.1677
   ACTB: PCC=-0.1925
   PLEC: PCC=-0.1958
   TNFRSF12A: PCC=-0.1988
   TGFBI: PCC=-0.1994
   ITGA6: PCC=-0.2071
   MMP1: PCC=-0.2180
   SERPINE1: PCC=-0.2304
   ITGA3: PCC=-0.2357
   ANXA5: PCC=-0.2717

7. 결과 저장
✅ 저장 완료:
   - eval_loki_top300_common_sids.npy
   - eval_loki_top300_common_genes.npy
   - eval_loki_top300_gene_pcc.npy
   - eval_loki_top300_gene_mae.npy

✅ Loki Top 300 평가 완료!


In [4]:
# =========================
# 7. (추가) Violin plot용 raw matrix 저장
# =========================
print("\n" + "="*60)
print("8. Violin plot용 matrix 저장")
print("="*60)

METHOD_NAME = "baseline_top300"   # 너 실험 이름에 맞게 바꿔도 됨

# ✅ 이 코드에서의 raw matrices
Y_true = rna_array.astype(np.float32)   # (n_slides, n_genes)
Y_pred = pred_array.astype(np.float32)  # (n_slides, n_genes)

np.save("Y_true_top300.npy", Y_true)
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred)

# gene / slide order도 같이 저장 (중요)
np.save("gene_names_top300.npy", np.array(common_genes, dtype=object))
np.save("slide_ids_top300.npy", np.array(sid_list, dtype=object))

print("✅ Saved for violin plot:")
print(f" - Y_true_top300.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names_top300.npy: {len(common_genes)} genes")
print(f" - slide_ids_top300.npy: {len(sid_list)} slides")



8. Violin plot용 matrix 저장
✅ Saved for violin plot:
 - Y_true_top300.npy: (331, 290)
 - Y_pred_baseline_top300.npy: (331, 290)
 - gene_names_top300.npy: 290 genes
 - slide_ids_top300.npy: 331 slides


In [ ]:
## 실험 2: Top-K=128 + ReLU 예측

Top-K 필터링을 추가하여 더 안정적인 예측을 수행합니다.


In [5]:
import numpy as np
from pathlib import Path
import sys

# Loki 모듈 import
loki_path = "/project_antwerp/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        """Loki PredEx 함수 직접 구현"""
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression_loki_top300_topk128")  # Top-K + ReLU 결과 저장
OUT_DIR.mkdir(exist_ok=True)

EPS = 1e-8
PATCH_CHUNK = 256
TOPK = 128  # Top-K 개수

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

print(f"✅ 데이터 로드 완료: {Ntrain}개 학습 샘플, {G}개 유전자")

# =========================
# Top 300 genes 인덱스 로드 및 train_expr 필터링
# =========================
genes_all = [line.strip() for line in open("/project_antwerp/hbae/data/all_shared_genes.txt") if line.strip()]
top_genes = [line.strip() for line in open("/project_antwerp/hbae/script/top300_genes.txt") if line.strip()]

gene2idx = {g: i for i, g in enumerate(genes_all)}
top_idx = np.array([gene2idx[g] for g in top_genes], dtype=np.int64)

# 🚀 성능 최적화: train_expr를 미리 Top 300개만 필터링
train_expr_top300 = train_expr[:, top_idx]  # (Ntrain, 300)

print(f"✅ Top 300 genes 인덱스 로드: {len(top_idx)}개")
print(f"✅ train_expr 필터링 완료: {train_expr.shape} → {train_expr_top300.shape}")

# =========================
# Slide loop - Top-K + ReLU 방식
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    FORCE_RECOMPUTE = False
    
    if not FORCE_RECOMPUTE and out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id} (Top-K={TOPK} + ReLU)")

    # 이미지 임베딩 로드 및 정규화
    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    # 패치별로 청크 단위 처리
    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        img_chunk = img_emb[p0:p1]  # (chunk_size, 768)
        
        # Similarity 계산
        similarity = img_chunk @ train_text_emb.T  # (chunk_size, Ntrain)
        
        # Top-K 선택
        topk_idx = np.argpartition(similarity, -TOPK, axis=1)[:, -TOPK:]  # (chunk_size, TOPK)
        topk_similarity = np.take_along_axis(similarity, topk_idx, axis=1)  # (chunk_size, TOPK)
        
        # ReLU 적용
        topk_similarity = np.maximum(topk_similarity, 0.0)
        
        # Top-K train_expr 선택
        topk_train_expr = train_expr_top300[topk_idx]  # (chunk_size, TOPK, 300)
        
        # 가중 평균 계산 (Top-K만 사용)
        weights = topk_similarity.sum(axis=1, keepdims=True) + EPS  # (chunk_size, 1)
        weighted_sum = (topk_similarity[:, :, None] * topk_train_expr).sum(axis=1)  # (chunk_size, 300)
        pred_chunk = weighted_sum / weights  # (chunk_size, 300)
        
        pred_chunks.append(pred_chunk)

    pred_expr = np.vstack(pred_chunks)  # (P, 300)
    
    # 저장
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"
    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape} (Top-K={TOPK} + ReLU)")

print("\n✅ Top-K + ReLU 방식으로 전체 슬라이드 예측 완료")
print(f"📁 결과 저장 위치: {OUT_DIR}")


✅ Loki 모듈 로드 성공
Train text: (154763, 768)
Train expr: (154763, 12009)
✅ 데이터 로드 완료: 154763개 학습 샘플, 12009개 유전자
✅ Top 300 genes 인덱스 로드: 300개
✅ train_expr 필터링 완료: (154763, 12009) → (154763, 300)

🔹 Processing slide: TCGA-4P-AA8J-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (11834, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-4074-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (1033, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-4077-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (1749, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-5151-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (1444, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-5153-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (2777, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-6870-01Z-00-DX1 (Top-K=128 + ReLU)
✅ Saved predicted expression: (157, 300) (Top-K=128 + ReLU)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1 (Top-K=128 + R

In [ ]:
## 실험 3: Z-score 정규화 후 평가

Gene-wise z-score 정규화를 적용한 후 PCC를 재계산합니다.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

# =========================
# 경로 설정
# =========================
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PATCH_PRED_DIR = Path("tcga_predicted_expression_loki_top300_topk128")  # 패치 레벨 예측 결과
TOP300_GENES_FILE = "/project_antwerp/hbae/script/top300_genes.txt"

# =========================
# 정규화 함수
# =========================
def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

# =========================
# 1. TCGA RNA-seq 데이터 로드
# =========================
print("="*60)
print("1. TCGA RNA-seq 데이터 로드")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

# wsi_file_name 컬럼에서 slide_id 추출
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])
if 'wsi_file_name' in rna_df.columns:
    rna_df = rna_df.drop(columns=['wsi_file_name'])

# 숫자 컬럼만 선택
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# rna_ prefix 제거 및 정규화
rna_gene_names = [c.replace("rna_", "") for c in rna_expr_df.columns]
rna_gene_norm = [norm_gene(g) for g in rna_gene_names]

# RNA gene map
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")

# =========================
# 2. Top 300 genes 로드
# =========================
print("\n" + "="*60)
print("2. Top 300 genes 로드")
print("="*60)

top_genes = [line.strip() for line in open(TOP300_GENES_FILE) if line.strip()]
top_genes_norm = [norm_gene(g) for g in top_genes]

print(f"Top 300 genes: {len(top_genes)}")

# =========================
# 3. 공통 genes 찾기
# =========================
print("\n" + "="*60)
print("3. 공통 genes 찾기")
print("="*60)

common_genes = sorted(set(top_genes_norm) & set(rna_gene_map.keys()))

if len(common_genes) < len(top_genes_norm):
    print(f"⚠️ 경고: Top 300 중 {len(top_genes_norm) - len(common_genes)}개가 RNA-seq에 없음")
    print(f"   최종 비교 genes: {len(common_genes)}개")

# 인덱스 매핑
rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([top_genes_norm.index(g) for g in common_genes], dtype=np.int64)

print(f"✅ 공통 genes: {len(common_genes)}개")

# =========================
# 4. 공통 slides 찾기
# =========================
print("\n" + "="*60)
print("4. 공통 slides 찾기")
print("="*60)

pred_files = sorted(PATCH_PRED_DIR.glob("*.npy"))
pred_sids_raw = [p.stem for p in pred_files]
pred_sids = [norm_sid(sid) for sid in pred_sids_raw]
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))

print(f"Pred slides: {len(pred_sid_set)}")
print(f"Common slides: {len(common_sids)}")

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0입니다.")

# =========================
# 5. 예측값과 실제값 로드 (패치 → 슬라이드 집계)
# =========================
print("\n" + "="*60)
print("5. 예측값과 실제값 로드 (패치 → 슬라이드 집계)")
print("="*60)

pred_list = []
rna_list = []
sid_list = []

for sid in tqdm(common_sids):
    # 예측값 로드 (패치 레벨)
    pred_file = None
    for p in pred_files:
        if norm_sid(p.stem) == sid:
            pred_file = p
            break
    
    if pred_file is None or not pred_file.exists():
        continue
    
    patch_pred = np.load(pred_file)  # (num_patches, 300) 또는 (300,)
    
    # 패치 레벨인 경우 슬라이드 레벨로 집계
    if patch_pred.ndim == 2:
        slide_pred = patch_pred.mean(axis=0)  # (300,)
    else:
        slide_pred = patch_pred  # 이미 슬라이드 레벨
    
    # 공통 genes만 선택
    if len(slide_pred) == 300:
        pred_selected = slide_pred[pred_idx]  # 공통 genes만 선택
    else:
        # 예상치 못한 크기인 경우
        print(f"⚠️ 경고: {sid}의 예측값 크기가 예상과 다름: {slide_pred.shape}")
        continue
    
    # 실제값 로드
    rna_idx_slide = rna_slide_map[sid]
    rna = rna_expr_df.iloc[rna_idx_slide, rna_idx].values  # 공통 genes만 선택
    
    pred_list.append(pred_selected)
    rna_list.append(rna)
    sid_list.append(sid)

pred_array = np.array(pred_list)  # (num_slides, num_common_genes)
rna_array = np.array(rna_list)   # (num_slides, num_common_genes)

print(f"✅ 로드 완료: {pred_array.shape}")

# =========================
# 6. 평가 메트릭 계산
# =========================
print("\n" + "="*60)
print("6. 평가 메트릭 계산")
print("="*60)

# Overall PCC (flatten)
pred_flat = pred_array.flatten()
rna_flat = rna_array.flatten()
overall_pcc, _ = pearsonr(pred_flat, rna_flat)
overall_spearman, _ = spearmanr(pred_flat, rna_flat)

# Gene-wise PCC
gene_pccs = []
gene_spearmans = []
gene_maes = []

for i in range(len(common_genes)):
    gene_pred = pred_array[:, i]
    gene_rna = rna_array[:, i]
    
    # PCC
    pcc, _ = pearsonr(gene_pred, gene_rna)
    if not np.isnan(pcc):
        gene_pccs.append(pcc)
    
    # Spearman
    spearman, _ = spearmanr(gene_pred, gene_rna)
    if not np.isnan(spearman):
        gene_spearmans.append(spearman)
    
    # MAE
    mae = np.mean(np.abs(gene_pred - gene_rna))
    gene_maes.append(mae)

gene_pccs = np.array(gene_pccs)
gene_spearmans = np.array(gene_spearmans)
gene_maes = np.array(gene_maes)

print(f"📌 Overall PCC: {overall_pcc:.4f}")
print(f"📌 Overall Spearman: {overall_spearman:.4f}")
print(f"📌 Gene-wise PCC mean: {gene_pccs.mean():.4f}")
print(f"📌 Gene-wise PCC std: {gene_pccs.std():.4f}")
print(f"📌 Gene-wise Spearman mean: {gene_spearmans.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_maes.mean():.4f}")

# Top/Bottom PCC genes
gene_pcc_dict = {common_genes[i]: gene_pccs[i] for i in range(len(common_genes)) if i < len(gene_pccs) and not np.isnan(gene_pccs[i])}
sorted_genes = sorted(gene_pcc_dict.items(), key=lambda x: x[1], reverse=True)

print(f"\n✅ Top 10 PCC genes:")
for gene, pcc in sorted_genes[:10]:
    print(f"   {gene}: PCC={pcc:.4f}")

print(f"\n❌ Bottom 10 PCC genes:")
for gene, pcc in sorted_genes[-10:]:
    print(f"   {gene}: PCC={pcc:.4f}")

# =========================
# 7. 결과 저장
# =========================
print("\n" + "="*60)
print("7. 결과 저장")
print("="*60)

np.save("eval_loki_top300_topk128_common_sids.npy", np.array(common_sids))
np.save("eval_loki_top300_topk128_common_genes.npy", np.array(common_genes))
np.save("eval_loki_top300_topk128_gene_pcc.npy", gene_pccs)
np.save("eval_loki_top300_topk128_gene_spearman.npy", gene_spearmans)
np.save("eval_loki_top300_topk128_gene_mae.npy", gene_maes)

print("✅ 저장 완료:")
print("   - eval_loki_top300_topk128_common_sids.npy")
print("   - eval_loki_top300_topk128_common_genes.npy")
print("   - eval_loki_top300_topk128_gene_pcc.npy")
print("   - eval_loki_top300_topk128_gene_spearman.npy")
print("   - eval_loki_top300_topk128_gene_mae.npy")

print("\n" + "="*60)
print("✅ Top-K=128 + ReLU 평가 완료!")
print("="*60)


1. TCGA RNA-seq 데이터 로드
RNA slides: 461, RNA genes: 24778

2. Top 300 genes 로드
Top 300 genes: 300

3. 공통 genes 찾기
⚠️ 경고: Top 300 중 10개가 RNA-seq에 없음
   최종 비교 genes: 290개
✅ 공통 genes: 290개

4. 공통 slides 찾기
Pred slides: 331
Common slides: 331

5. 예측값과 실제값 로드 (패치 → 슬라이드 집계)


100%|██████████| 331/331 [00:05<00:00, 62.14it/s]


✅ 로드 완료: (331, 290)

6. 평가 메트릭 계산
📌 Overall PCC: 0.4519
📌 Overall Spearman: 0.3580
📌 Gene-wise PCC mean: 0.0105
📌 Gene-wise PCC std: 0.0806
📌 Gene-wise Spearman mean: 0.0111
📌 Gene-wise MAE mean: 3.1222

✅ Top 10 PCC genes:
   CD44: PCC=0.2415
   ACTN1: PCC=0.2269
   MMP1: PCC=0.2064
   ANXA2: PCC=0.2004
   PRNP: PCC=0.1939
   GLUL: PCC=0.1862
   KRT15: PCC=0.1830
   ITGA6: PCC=0.1798
   PLS3: PCC=0.1746
   GJA1: PCC=0.1742

❌ Bottom 10 PCC genes:
   TNC: PCC=-0.1340
   PPIB: PCC=-0.1417
   PITX1: PCC=-0.1424
   ACTB: PCC=-0.1436
   FGFBP1: PCC=-0.1533
   ANXA5: PCC=-0.1686
   PGAM1: PCC=-0.1694
   HIF1A: PCC=-0.1811
   SERPINE1: PCC=-0.1813
   CAP1: PCC=-0.2064

7. 결과 저장
✅ 저장 완료:
   - eval_loki_top300_topk128_common_sids.npy
   - eval_loki_top300_topk128_common_genes.npy
   - eval_loki_top300_topk128_gene_pcc.npy
   - eval_loki_top300_topk128_gene_spearman.npy
   - eval_loki_top300_topk128_gene_mae.npy

✅ Top-K=128 + ReLU 평가 완료!


In [3]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

# =========================
# 7. (추가) Violin plot용 raw matrix 저장
# =========================
print("\n" + "="*60)
print("8. Violin plot용 matrix 저장")
print("="*60)

METHOD_NAME = "topk_relu_top300"   # 너 실험 이름에 맞게 바꿔도 됨

# ✅ 이 코드에서의 raw matrices
Y_true = rna_array.astype(np.float32)   # (n_slides, n_genes)
Y_pred = pred_array.astype(np.float32)  # (n_slides, n_genes)

np.save("Y_true_topk_relu_top300.npy", Y_true)
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred)

# gene / slide order도 같이 저장 (중요)
np.save("gene_names_topk_relu_top300.npy", np.array(common_genes, dtype=object))
np.save("slide_ids_topk_relu_top300.npy", np.array(sid_list, dtype=object))

print("✅ Saved for violin plot:")
print(f" - Y_true_top300.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names_top300.npy: {len(common_genes)} genes")
print(f" - slide_ids_top300.npy: {len(sid_list)} slides")



8. Violin plot용 matrix 저장
✅ Saved for violin plot:
 - Y_true_top300.npy: (331, 290)
 - Y_pred_topk_relu_top300.npy: (331, 290)
 - gene_names_top300.npy: 290 genes
 - slide_ids_top300.npy: 331 slides


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import sys

# Loki 모듈 import
loki_path = "/project_antwerp/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        """Loki PredEx 함수 직접 구현"""
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# 경로 설정
# =========================
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
EMB_DIR = Path("/project_antwerp/hbae/script/embedding/tcga_embeddings")  # tile-level image embeddings
TOP300_GENES_FILE = "/project_antwerp/hbae/script/top300_genes.txt"
TRAIN_TEXT_EMB = "train_text_embedding.npy"
TRAIN_EXPR = "/project_antwerp/hbae/data/combined_expression_matrix.npy"

# =========================
# 정규화 함수
# =========================
def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

# =========================
# 1. Training 데이터 로드
# =========================
print("="*60)
print("1. Training 데이터 로드")
print("="*60)

train_text_emb = np.load(TRAIN_TEXT_EMB).astype(np.float32)  # (Ntrain, 768)
train_expr = np.load(TRAIN_EXPR).astype(np.float32)  # (Ntrain, G)

EPS = 1e-8
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

print(f"Train text embedding: {train_text_emb.shape}")
print(f"Train expression: {train_expr.shape}")

# =========================
# 2. Top 300 genes 로드 및 매핑
# =========================
print("\n" + "="*60)
print("2. Top 300 genes 로드")
print("="*60)

top_genes = [line.strip() for line in open(TOP300_GENES_FILE) if line.strip()]
top_genes_norm = [norm_gene(g) for g in top_genes]

# 전체 gene list에서 top 300 genes의 인덱스 찾기
# train_expr의 gene 순서는 all_shared_genes.txt와 동일하다고 가정
ALL_SHARED_GENES = "/project_antwerp/hbae/data/all_shared_genes.txt"
all_genes = [line.strip() for line in open(ALL_SHARED_GENES) if line.strip()]
all_genes_norm = [norm_gene(g) for g in all_genes]

# top 300 genes의 인덱스 (전체 gene list 기준)
top300_indices = []
for g in top_genes_norm:
    if g in all_genes_norm:
        top300_indices.append(all_genes_norm.index(g))
    else:
        print(f"⚠️ 경고: {g}가 전체 gene list에 없음")

print(f"Top 300 genes: {len(top_genes)}")
print(f"매핑된 인덱스: {len(top300_indices)}")

# train_expr에서 top 300 genes만 선택
train_expr_top300 = train_expr[:, top300_indices]  # (Ntrain, 300)

# =========================
# 3. TCGA RNA-seq 데이터 로드
# =========================
print("\n" + "="*60)
print("3. TCGA RNA-seq 데이터 로드")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

# wsi_file_name 컬럼에서 slide_id 추출
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])
if 'wsi_file_name' in rna_df.columns:
    rna_df = rna_df.drop(columns=['wsi_file_name'])

# 숫자 컬럼만 선택
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# rna_ prefix 제거 및 정규화
rna_gene_names = [c.replace("rna_", "") for c in rna_expr_df.columns]
rna_gene_norm = [norm_gene(g) for g in rna_gene_names]

# RNA gene map
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")

# =========================
# 4. 공통 genes 찾기
# =========================
print("\n" + "="*60)
print("4. 공통 genes 찾기")
print("="*60)

common_genes = sorted(set(top_genes_norm) & set(rna_gene_map.keys()))

if len(common_genes) < len(top_genes_norm):
    print(f"⚠️ 경고: Top 300 중 {len(top_genes_norm) - len(common_genes)}개가 RNA-seq에 없음")

# 인덱스 매핑
rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
# top 300 genes 내에서의 인덱스
pred_idx_in_top300 = []
for g in common_genes:
    if g in top_genes_norm:
        pred_idx_in_top300.append(top_genes_norm.index(g))
    else:
        print(f"⚠️ 경고: {g}가 top 300에 없음")
pred_idx_in_top300 = np.array(pred_idx_in_top300, dtype=np.int64)

print(f"✅ 공통 genes: {len(common_genes)}개")

# =========================
# 5. 공통 slides 찾기
# =========================
print("\n" + "="*60)
print("5. 공통 slides 찾기")
print("="*60)

emb_files = sorted(EMB_DIR.glob("*.npy"))
emb_sids_raw = [p.stem for p in emb_files]
emb_sids = [norm_sid(sid) for sid in emb_sids_raw]
emb_sid_set = set(emb_sids)

common_sids = sorted(emb_sid_set & set(rna_slide_map.keys()))

print(f"Embedding slides: {len(emb_sid_set)}")
print(f"Common slides: {len(common_sids)}")

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0입니다.")

# =========================
# 6. 예측값 계산 (튜터 방식: tile-level similarity → weighted average → slide 평균)
# =========================
print("\n" + "="*60)
print("6. 예측값 계산 (tile-level similarity → weighted average → slide 평균)")
print("="*60)

pred_list = []
rna_list = []
sid_list = []

for sid in tqdm(common_sids):
    # Tile-level image embedding 로드
    emb_file = None
    for p in emb_files:
        if norm_sid(p.stem) == sid:
            emb_file = p
            break
    
    if emb_file is None or not emb_file.exists():
        continue
    
    # 🔑 Tile-level image embedding 로드
    tile_emb = np.load(emb_file).astype(np.float32)  # (num_tiles, 768)
    tile_emb /= (np.linalg.norm(tile_emb, axis=1, keepdims=True) + EPS)
    
    # 🔑 Cosine similarity matrix 계산 (튜터 방식)
    # tile-level image embedding @ text embedding.T
    similarity = tile_emb @ train_text_emb.T  # (num_tiles, Ntrain)
    
    # 🔑 Weighted average (predict_st_gene_expr 사용)
    # training에 썼던 spatial transcriptomics expression data로 weighted average
    tile_pred = loki.predex.predict_st_gene_expr(similarity, train_expr_top300)  # (num_tiles, 300)
    
    # 🔑 Slide-level prediction: tile-level prediction의 평균
    slide_pred = tile_pred.mean(axis=0)  # (300,)
    
    # 공통 genes만 선택
    pred_selected = slide_pred[pred_idx_in_top300]  # (len(common_genes),)
    
    # 실제값 로드
    rna_idx_slide = rna_slide_map[sid]
    rna = rna_expr_df.iloc[rna_idx_slide, rna_idx].values  # 공통 genes만 선택
    
    pred_list.append(pred_selected)
    rna_list.append(rna)
    sid_list.append(sid)

pred_array = np.array(pred_list)  # (num_slides, num_common_genes)
rna_array = np.array(rna_list)   # (num_slides, num_common_genes)

print(f"✅ 예측 완료: {pred_array.shape}")

# =========================
# 7. 평가 메트릭 계산
# =========================
print("\n" + "="*60)
print("7. 평가 메트릭 계산")
print("="*60)

# Overall PCC (flatten)
pred_flat = pred_array.flatten()
rna_flat = rna_array.flatten()
overall_pcc, _ = pearsonr(pred_flat, rna_flat)
overall_spearman, _ = spearmanr(pred_flat, rna_flat)

# Gene-wise PCC
gene_pccs = []
gene_spearmans = []
gene_maes = []

for i in range(len(common_genes)):
    gene_pred = pred_array[:, i]
    gene_rna = rna_array[:, i]
    
    # PCC
    pcc, _ = pearsonr(gene_pred, gene_rna)
    if not np.isnan(pcc):
        gene_pccs.append(pcc)
    
    # Spearman
    spearman, _ = spearmanr(gene_pred, gene_rna)
    if not np.isnan(spearman):
        gene_spearmans.append(spearman)
    
    # MAE
    mae = np.mean(np.abs(gene_pred - gene_rna))
    gene_maes.append(mae)

gene_pccs = np.array(gene_pccs)
gene_spearmans = np.array(gene_spearmans)
gene_maes = np.array(gene_maes)

print(f"📌 Overall PCC: {overall_pcc:.4f}")
print(f"📌 Overall Spearman: {overall_spearman:.4f}")
print(f"📌 Gene-wise PCC mean: {gene_pccs.mean():.4f}")
print(f"📌 Gene-wise PCC std: {gene_pccs.std():.4f}")
print(f"📌 Gene-wise Spearman mean: {gene_spearmans.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_maes.mean():.4f}")

# Top/Bottom PCC genes
gene_pcc_dict = {common_genes[i]: gene_pccs[i] for i in range(len(common_genes)) if i < len(gene_pccs) and not np.isnan(gene_pccs[i])}
sorted_genes = sorted(gene_pcc_dict.items(), key=lambda x: x[1], reverse=True)

print(f"\n✅ Top 10 PCC genes:")
for gene, pcc in sorted_genes[:10]:
    print(f"   {gene}: PCC={pcc:.4f}")

print(f"\n❌ Bottom 10 PCC genes:")
for gene, pcc in sorted_genes[-10:]:
    print(f"   {gene}: PCC={pcc:.4f}")

# =========================
# 8. 결과 저장
# =========================
print("\n" + "="*60)
print("8. 결과 저장")
print("="*60)

np.save("eval_loki_top300_tutor_method_common_sids.npy", np.array(sid_list))
np.save("eval_loki_top300_tutor_method_common_genes.npy", np.array(common_genes))
np.save("eval_loki_top300_tutor_method_gene_pcc.npy", gene_pccs)
np.save("eval_loki_top300_tutor_method_gene_spearman.npy", gene_spearmans)
np.save("eval_loki_top300_tutor_method_gene_mae.npy", gene_maes)

print("✅ 저장 완료:")
print("   - eval_loki_top300_tutor_method_common_sids.npy")
print("   - eval_loki_top300_tutor_method_common_genes.npy")
print("   - eval_loki_top300_tutor_method_gene_pcc.npy")
print("   - eval_loki_top300_tutor_method_gene_spearman.npy")
print("   - eval_loki_top300_tutor_method_gene_mae.npy")

print("\n" + "="*60)
print("✅ 튜터 방식 평가 완료!")
print("="*60)



✅ Loki 모듈 로드 성공
1. Training 데이터 로드
Train text embedding: (154763, 768)
Train expression: (154763, 12009)

2. Top 300 genes 로드
Top 300 genes: 300
매핑된 인덱스: 300

3. TCGA RNA-seq 데이터 로드
RNA slides: 461, RNA genes: 24778

4. 공통 genes 찾기
⚠️ 경고: Top 300 중 10개가 RNA-seq에 없음
✅ 공통 genes: 290개

5. 공통 slides 찾기
Embedding slides: 331
Common slides: 331

6. 예측값 계산 (tile-level similarity → weighted average → slide 평균)


100%|██████████| 331/331 [19:42<00:00,  3.57s/it]


✅ 예측 완료: (331, 290)

7. 평가 메트릭 계산
📌 Overall PCC: 0.5489
📌 Overall Spearman: 0.4580
📌 Gene-wise PCC mean: -0.0017
📌 Gene-wise PCC std: 0.0929
📌 Gene-wise Spearman mean: 0.0030
📌 Gene-wise MAE mean: 3.1348

✅ Top 10 PCC genes:
   GJA1: PCC=0.2203
   DSTN: PCC=0.2119
   KRT15: PCC=0.1960
   CD44: PCC=0.1893
   ANXA2: PCC=0.1858
   HIF1A: PCC=0.1738
   PGAM1: PCC=0.1668
   KRT6B: PCC=0.1611
   AHNAK2: PCC=0.1603
   TUBB6: PCC=0.1568

❌ Bottom 10 PCC genes:
   MSN: PCC=-0.1676
   ACTB: PCC=-0.1925
   PLEC: PCC=-0.1956
   TNFRSF12A: PCC=-0.1987
   TGFBI: PCC=-0.1994
   ITGA6: PCC=-0.2069
   MMP1: PCC=-0.2180
   SERPINE1: PCC=-0.2304
   ITGA3: PCC=-0.2355
   ANXA5: PCC=-0.2716

8. 결과 저장
✅ 저장 완료:
   - eval_loki_top300_tutor_method_common_sids.npy
   - eval_loki_top300_tutor_method_common_genes.npy
   - eval_loki_top300_tutor_method_gene_pcc.npy
   - eval_loki_top300_tutor_method_gene_spearman.npy
   - eval_loki_top300_tutor_method_gene_mae.npy

✅ 튜터 방식 평가 완료!


In [5]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

# METHOD_NAME = "baseline"  
METHOD_NAME = "topk_relu_top300_tutor"
# METHOD_NAME = "linear"

np.save("Y_true.npy", Y_true.astype(np.float32))
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred.astype(np.float32))

# 이미 gene / slide 정보는 저장돼 있으므로 그대로 사용
np.save("gene_names.npy", np.array(common_genes, dtype=object))
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))

print("\n✅ Saved files:")
print(f" - Y_true.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names.npy: {len(common_genes)} genes")
print(f" - eval_common_sids.npy: {len(common_sids)} slides")



✅ Saved files:
 - Y_true.npy: (331, 290)
 - Y_pred_topk_relu_top300_tutor.npy: (331, 290)
 - gene_names.npy: 290 genes
 - eval_common_sids.npy: 331 slides


In [6]:
## Gene별 표준편차 분석

예측값과 실제값의 gene별 표준편차를 비교합니다.


SyntaxError: invalid syntax (2131509002.py, line 3)

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =========================
# 경로 설정 (평가 코드와 동일)
# =========================
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_DIR = Path("tcga_predicted_expression_loki_top300_topk128")  # 슬라이드 레벨 예측 결과
TOP300_GENES_FILE = "/project_antwerp/hbae/script/top300_genes.txt"

# =========================
# 정규화 함수
# =========================
def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

# =========================
# 데이터 로드 (평가 코드와 동일한 로직)
# =========================
rna_df = pd.read_csv(TCGA_RNA_CSV)

# wsi_file_name 컬럼에서 slide_id 추출
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])
if 'wsi_file_name' in rna_df.columns:
    rna_df = rna_df.drop(columns=['wsi_file_name'])

# 숫자 컬럼만 선택
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# rna_ prefix 제거 및 정규화
rna_gene_names = [c.replace("rna_", "") for c in rna_expr_df.columns]
rna_gene_norm = [norm_gene(g) for g in rna_gene_names]

# RNA gene map
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

# Top 300 genes 로드
top_genes = [line.strip() for line in open(TOP300_GENES_FILE) if line.strip()]
top_genes_norm = [norm_gene(g) for g in top_genes]

# 공통 genes 찾기
common_genes = sorted(set(top_genes_norm) & set(rna_gene_map.keys()))

# 인덱스 매핑
rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([top_genes_norm.index(g) for g in common_genes], dtype=np.int64)

# 공통 slides 찾기
pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids_raw = [p.stem for p in pred_files]
pred_sids = [norm_sid(sid) for sid in pred_sids_raw]
pred_sid_set = set(pred_sids)
common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))

# 예측값과 실제값 로드
pred_list = []
rna_list = []

for sid in tqdm(common_sids):
    # 예측값 로드
    pred_file = None
    for p in pred_files:
        if norm_sid(p.stem) == sid:
            pred_file = p
            break
    
    if pred_file is None or not pred_file.exists():
        continue
    
    pred_raw = np.load(pred_file)
    
    # 패치 레벨인 경우 슬라이드 레벨로 집계
    if pred_raw.ndim == 2:
        pred = pred_raw.mean(axis=0)
    else:
        pred = pred_raw
    
    if len(pred) != 300:
        continue
    
    pred_selected = pred[pred_idx]
    
    # 실제값 로드
    rna_idx_slide = rna_slide_map[sid]
    rna = rna_expr_df.iloc[rna_idx_slide, rna_idx].values
    
    pred_list.append(pred_selected)
    rna_list.append(rna)

# 변수 정의
pred_slides = np.array(pred_list)  # (num_slides, num_common_genes)
true_slides = np.array(rna_list)   # (num_slides, num_common_genes)

print(f"✅ 데이터 로드 완료: {pred_slides.shape}")

# =========================
# Gene별 표준편차 분석
# =========================
pred_std_per_gene = pred_slides.std(axis=0)  # gene별 슬라이드 표준편차
true_std_per_gene = true_slides.std(axis=0)

print(f"\n📊 Gene별 표준편차 비교:")
print(f"   예측값 평균 표준편차: {pred_std_per_gene.mean():.4f}")
print(f"   실제값 평균 표준편차: {true_std_per_gene.mean():.4f}")
print(f"   비율 (예측/실제): {pred_std_per_gene.mean() / true_std_per_gene.mean():.4f}")

# 표준편차가 매우 낮은 genes (거의 변동 없음)
low_std_genes = np.where(pred_std_per_gene < 0.1)[0]
if len(low_std_genes) > 0:
    print(f"\n⚠️ 예측값 표준편차 < 0.1인 genes: {len(low_std_genes)}개")
    print(f"   예시: {[common_genes[i] for i in low_std_genes[:10]]}")

# =========================
# Gene mean 간 상관계수
# =========================
true_gene_means = true_slides.mean(axis=0)  # gene별 평균 발현
pred_gene_means = pred_slides.mean(axis=0)  # gene별 평균 발현

corr_gene_means = np.corrcoef(true_gene_means, pred_gene_means)[0, 1]

print(f"\n📊 Gene mean 간 상관계수:")
print(f"   Corr of gene means: {corr_gene_means:.4f}")


100%|██████████| 331/331 [00:05<00:00, 62.60it/s]

✅ 데이터 로드 완료: (331, 290)

📊 Gene별 표준편차 비교:
   예측값 평균 표준편차: 0.1957
   실제값 평균 표준편차: 0.7833
   비율 (예측/실제): 0.2499

⚠️ 예측값 표준편차 < 0.1인 genes: 60개
   예시: ['ACTN1', 'ANXA5', 'ARPC1B', 'ARPC5', 'ATP1A1', 'ATP2A2', 'ATP5F1B', 'ATP5F1D', 'ATP5IF1', 'ATP5PD']

📊 Gene mean 간 상관계수:
   Corr of gene means: 0.5787


In [8]:
# slide-level (331, 290) 이 이미 있다면
pred_std_per_gene = pred_slides.std(axis=0)  # gene별 슬라이드 표준편차
true_std_per_gene = true_slides.std(axis=0)

print(pred_std_per_gene.mean(), true_std_per_gene.mean())


0.1957319 0.78325805718915


In [9]:
corr_gene_means = np.corrcoef(true_slides.mean(axis=0), pred_slides.mean(axis=0))[0,1]
print("corr of gene means:", corr_gene_means)


corr of gene means: 0.5786888760004815


## Loki PredEx 방식으로 예측 (Loki 모듈 사용)

현재 Cell 14는 수동으로 Top-K + ReLU 가중치를 사용하지만, 
Loki의 공식 `predict_st_gene_expr` 함수를 사용하면 더 간단하고 표준적인 방식으로 예측할 수 있습니다.

**차이점:**
- **현재 방식**: Top-K(128) 선택 + ReLU(similarity) 가중치
- **Loki 방식**: 모든 similarity 사용 + 원본 similarity 가중치

**실행 방법 (gpulab):**
1. Cell 15 실행: 패치 레벨 예측 (Loki 방식)
   - 결과 저장: `tcga_predicted_expression_loki/`
   - 기존 파일이 있으면 자동으로 건너뜀 (시간 절약)
   - 강제 재처리: 코드 내 `FORCE_RECOMPUTE = True` 설정

2. Cell 17 실행: 슬라이드 레벨 예측 (Loki 방식)
   - 결과 저장: `tcga_pred_expr_loki/`
   - 기존 파일이 있으면 자동으로 건너뜀


## Loki PredEx 방식으로 예측 (Loki 모듈 사용)

현재 Cell 14는 수동으로 Top-K + ReLU 가중치를 사용하지만, 
Loki의 공식 `predict_st_gene_expr` 함수를 사용하면 더 간단하고 표준적인 방식으로 예측할 수 있습니다.

**차이점:**
- **현재 방식**: Top-K(128) 선택 + ReLU(similarity) 가중치
- **Loki 방식**: 모든 similarity 사용 + 원본 similarity 가중치

**실행 방법 (gpulab):**
1. Cell 15 실행: 패치 레벨 예측 (Loki 방식)
   - 결과 저장: `tcga_predicted_expression_loki/`
   - 기존 파일이 있으면 자동으로 건너뜀 (시간 절약)
   - 강제 재처리: 코드 내 `FORCE_RECOMPUTE = True` 설정

2. Cell 17 실행: 슬라이드 레벨 예측 (Loki 방식)
   - 결과 저장: `tcga_pred_expr_loki/`
   - 기존 파일이 있으면 자동으로 건너뜀


In [10]:
import numpy as np
from pathlib import Path
import sys

# Loki 모듈 import
loki_path = "/data/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        """Loki PredEx 함수 직접 구현"""
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression_loki")  # Loki 방식 결과 저장
OUT_DIR.mkdir(exist_ok=True)

EPS = 1e-8
PATCH_CHUNK = 256    # 메모리 효율을 위한 청크 크기

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize (cosine similarity = dot product)
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

print(f"✅ 데이터 로드 완료: {Ntrain}개 학습 샘플, {G}개 유전자")

# =========================
# Slide loop - Loki 방식
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    # 🔄 Resume 기능: 기존 파일이 있으면 건너뛰기
    # 강제로 재처리하려면 아래 주석을 해제하세요:
    # FORCE_RECOMPUTE = True
    FORCE_RECOMPUTE = False  # 기본값: 기존 결과 사용
    
    if not FORCE_RECOMPUTE and out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id} (Loki 방식)")

    # 이미지 임베딩 로드 및 정규화
    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    # 패치별로 청크 단위 처리 (메모리 효율)
    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        img_chunk = img_emb[p0:p1]  # (chunk_size, 768)
        
        # 🔑 Loki 방식: 모든 similarity 계산 (Top-K 없음)
        similarity = img_chunk @ train_text_emb.T  # (chunk_size, Ntrain)
        
        # 🔑 Loki PredEx 함수 사용
        pred_chunk = loki.predex.predict_st_gene_expr(similarity, train_expr)  # (chunk_size, G)
        pred_chunks.append(pred_chunk)

    pred_expr = np.vstack(pred_chunks)  # (P, G)
    
    # 저장
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"
    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape}")

print("\n✅ Loki 방식으로 전체 슬라이드 예측 완료")


✅ Loki 모듈 로드 성공
Train text: (154763, 768)
Train expr: (154763, 12009)
✅ 데이터 로드 완료: 154763개 학습 샘플, 12009개 유전자

🔹 Processing slide: TCGA-4P-AA8J-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (11834, 12009)

🔹 Processing slide: TCGA-BA-4074-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (1033, 12009)

🔹 Processing slide: TCGA-BA-4077-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (1749, 12009)

🔹 Processing slide: TCGA-BA-5151-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (1444, 12009)

🔹 Processing slide: TCGA-BA-5153-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (2777, 12009)

🔹 Processing slide: TCGA-BA-6870-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (157, 12009)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (6126, 12009)

🔹 Processing slide: TCGA-BA-A6D8-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (11278, 12009)

🔹 Processing slide: TCGA-BA-A6DA-01Z-00-DX1 (Loki 방식)
✅ Saved predicted expression: (9139, 12009)

이 코드는 **TCGA 슬라이드의 이미지 임베딩을 사용해 패치별 유전자 발현을 예측**합니다. Loki PredEx의 기본 방식을 사용합니다.

## 코드 동작 설명

### 1. 초기 설정 (1-32줄)
- Loki 모듈 import: `/data/hbae/Loki/src`에서 `loki.predex` 모듈 로드
- 경로 설정:
  - `EMB_DIR`: 이미지 임베딩이 저장된 디렉토리 (`tcga_embeddings/`)
  - `OUT_DIR`: 예측 결과 저장 디렉토리 (`tcga_predicted_expression_loki/`)

### 2. 학습 데이터 로드 (35-49줄)
- `train_text_embedding.npy`: 학습 데이터의 텍스트 임베딩 (유전자 이름 → 768차원 벡터)
  - Shape: `(Ntrain, 768)` 예: `(154,763, 768)`
- `combined_expression_matrix.npy`: 학습 데이터의 실제 유전자 발현값
  - Shape: `(Ntrain, G)` 예: `(154,763, 12009)`
- L2 정규화: 코사인 유사도 계산을 위해 텍스트 임베딩 정규화

### 3. 각 슬라이드 처리 (54-100줄)
각 슬라이드의 이미지 임베딩 파일을 순회하며:

1. 이미지 임베딩 로드
   - `tcga_embeddings/{slide_id}.npy` 로드
   - Shape: `(num_patches, 768)` 예: `(11834, 768)`
   - L2 정규화

2. 패치별 예측 (청크 단위 처리)
   - 메모리 효율을 위해 패치를 256개씩 청크로 나눔
   - 각 청크에 대해:
     - 유사도 계산: `img_chunk @ train_text_emb.T` → `(chunk_size, Ntrain)`
     - Loki PredEx 함수 호출: `loki.predex.predict_st_gene_expr(similarity, train_expr)`
       - 모든 similarity 값을 가중치로 사용 (Top-K 필터링 없음)
       - 음수 similarity도 포함
       - 가중합 후 정규화하여 예측값 생성

3. 결과 저장
   - 모든 청크를 합쳐 `(num_patches, G)` 형태의 예측값 생성
   - `tcga_predicted_expression_loki/{slide_id}.npy`로 저장

### 4. Resume 기능 (61-68줄)
- 이미 처리된 파일은 건너뜀
- `FORCE_RECOMPUTE = True`로 설정하면 강제 재처리

## 핵심 특징

1. 기본 Loki 방식 사용
   - Top-K 필터링 없음
   - 모든 similarity 값을 가중치로 사용 (음수 포함)
   - `loki.predex.predict_st_gene_expr` 함수 사용

2. 패치 레벨 예측
   - 각 이미지 패치마다 유전자 발현 예측
   - 결과: `(num_patches, 12009)` 형태

3. 메모리 효율
   - 청크 단위 처리로 대용량 슬라이드 처리 가능

## 결과

각 슬라이드마다:
- `tcga_predicted_expression_loki/{slide_id}.npy`
- Shape: `(num_patches, 12009)`
- 각 패치의 12,009개 유전자 발현 예측값

이 예측값은 이후 평가나 슬라이드 레벨 예측에 사용됩니다.

## 슬라이드 레벨 예측 (Loki 방식)

슬라이드의 모든 패치를 평균내어 하나의 슬라이드 임베딩을 만들고, 
이를 사용하여 슬라이드 레벨 유전자 발현을 예측합니다.


In [12]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# paths
emb_dir = Path("tcga_embeddings")          # slide patch embeddings
out_dir = Path("tcga_pred_expr_loki")      # Loki 방식 슬라이드 레벨 예측 결과
out_dir.mkdir(exist_ok=True)

# 이미 로드된 데이터 사용 (Cell 15에서 로드한 것)
# train_text_emb: (N_train, 768)
# train_expr    : (N_train, G)

def l2norm(x, axis=-1, eps=1e-8):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

# Loki 방식으로 슬라이드 레벨 예측
slide_files = sorted(emb_dir.glob("*.npy"))
print(f"Slides: {len(slide_files)}")

for p in tqdm(slide_files):
    slide_id = p.stem
    out_path = out_dir / f"{slide_id}.npy"
    
    # 🔄 Resume 기능: 기존 파일이 있으면 건너뛰기
    # 강제로 재처리하려면 아래 주석을 해제하세요:
    # FORCE_RECOMPUTE = True
    FORCE_RECOMPUTE = False  # 기본값: 기존 결과 사용
    
    if not FORCE_RECOMPUTE and out_path.exists():
        continue

    # 패치 임베딩 로드 및 정규화
    patch_emb = np.load(p)                 # (num_patches, 768)
    patch_emb = l2norm(patch_emb)          # normalize patches

    # 🔑 슬라이드 임베딩: 모든 패치의 평균
    slide_emb = patch_emb.mean(axis=0)     # (768,)
    slide_emb = l2norm(slide_emb)          # normalize slide embedding

    # 🔑 Loki 방식: 모든 similarity 계산
    similarity = slide_emb @ train_text_emb.T  # (N_train,)
    similarity = similarity.reshape(1, -1)      # (1, N_train) - Loki 함수는 2D 배열 기대

    # 🔑 Loki PredEx 함수 사용
    pred = loki.predex.predict_st_gene_expr(similarity, train_expr)  # (1, G)
    pred = pred[0]  # (G,) - 1D로 변환

    np.save(out_path, pred)

print(f"\n✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to {out_dir}")


Slides: 331


100%|██████████| 331/331 [00:00<00:00, 2569.15it/s]


✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to tcga_pred_expr_loki


## Loki 방식 결과 평가 (PCC 및 Gene-wise PCC)

Cell 15와 17에서 생성한 Loki 방식 예측 결과를 평가합니다.
- 슬라이드 레벨 예측 결과: `tcga_pred_expr_loki/`
- 실제 RNA-seq 데이터와 비교하여 PCC와 gene-wise PCC를 계산합니다.


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =========================
# 설정
# =========================
PRED_DIR = Path("tcga_pred_expr_loki")  # Loki 방식 슬라이드 레벨 예측 결과
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"
HEG_GENE_LIST = "/project_antwerp/hbae/data/high_expression_genes_all.txt"

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

# =========================
# 1) RNA 데이터 로딩 및 정리
# =========================
print("="*60)
print("1. TCGA RNA 데이터 로딩")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

# slide_id 정규화
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

# 숫자 컬럼만 선택
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# RNA gene 이름: rna_ prefix 제거 및 정규화
rna_gene_names = np.array([c.replace("rna_", "") for c in rna_expr_df.columns], dtype=object)
rna_gene_norm = np.array([norm_gene(g) for g in rna_gene_names], dtype=object)

# RNA gene map (중복 있으면 첫번째만)
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")

# =========================
# 2) PredEx gene list 로딩
# =========================
print("\n" + "="*60)
print("2. PredEx gene list 로딩")
print("="*60)

pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_gene_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

pred_gene_map = {}
for i, g in enumerate(pred_gene_norm):
    if g not in pred_gene_map:
        pred_gene_map[g] = i

print(f"PredEx genes: {len(pred_gene_norm)}")

# =========================
# 3) HEG gene list 로딩
# =========================
print("\n" + "="*60)
print("3. HEG gene list 로딩")
print("="*60)

with open(HEG_GENE_LIST, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]
heg_genes = [norm_gene(g) for g in heg_genes]

print(f"HEG genes: {len(heg_genes)}")

# =========================
# 4) 공통 gene 및 slide 교집합
# =========================
print("\n" + "="*60)
print("4. 공통 gene 및 slide 매칭")
print("="*60)

common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print(f"✅ Common genes: {len(common_genes)}")
print(f"Example: {common_genes[:10]}")

rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)

# 공통 slide 교집합
pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids = [p.stem.split(".")[0] for p in pred_files]
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))
print(f"\nPred slides: {len(pred_sid_set)}")
print(f"Common slides: {len(common_sids)}")

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0입니다. 경로나 파일 이름을 확인하세요.")

# =========================
# 5) Y_true, Y_pred 만들기
# =========================
print("\n" + "="*60)
print("5. 평가 행렬 생성")
print("="*60)

N = len(common_sids)
G = len(common_genes)

Y_true = np.zeros((N, G), dtype=np.float32)
Y_pred = np.zeros((N, G), dtype=np.float32)

pred_path_map = {p.stem.split(".")[0]: p for p in pred_files}

for j, sid in enumerate(tqdm(common_sids, desc="Building matrices")):
    # RNA
    ridx = rna_slide_map[sid]
    rna_row = rna_expr_df.iloc[ridx].values.astype(np.float32)
    Y_true[j] = rna_row[rna_idx]

    # Pred (Loki 방식)
    pred_vec = np.load(pred_path_map[sid]).astype(np.float32)
    if pred_vec.ndim != 1:
        raise ValueError(f"{sid}: pred 벡터가 1D가 아님: {pred_vec.shape}")
    Y_pred[j] = pred_vec[pred_idx]

print(f"Y_true: {Y_true.shape}, Y_pred: {Y_pred.shape}")

# =========================
# 6) Overall PCC (flatten)
# =========================
print("\n" + "="*60)
print("6. 평가 메트릭 계산")
print("="*60)

x = Y_true.reshape(-1).astype(np.float64)
y = Y_pred.reshape(-1).astype(np.float64)

x = x - x.mean()
y = y - y.mean()
overall_pcc = (x @ y) / (np.sqrt((x @ x) * (y @ y)) + 1e-12)
print(f"📌 Overall PCC (flatten): {overall_pcc:.4f}")

# =========================
# 7) Gene-wise PCC (벡터화)
# =========================
Xt = Y_true.astype(np.float64)
Yp = Y_pred.astype(np.float64)

Xt_mean = Xt.mean(axis=0, keepdims=True)
Yp_mean = Yp.mean(axis=0, keepdims=True)

Xt_c = Xt - Xt_mean
Yp_c = Yp - Yp_mean

cov = (Xt_c * Yp_c).sum(axis=0)
stdx = np.sqrt((Xt_c * Xt_c).sum(axis=0))
stdy = np.sqrt((Yp_c * Yp_c).sum(axis=0))

gene_pcc = cov / (stdx * stdy + 1e-12)
gene_pcc[np.isnan(gene_pcc)] = 0.0

gene_mae = np.mean(np.abs(Y_true - Y_pred), axis=0)

print(f"📌 Gene-wise PCC mean: {gene_pcc.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_mae.mean():.4f}")

# top/bottom genes
topk = 20
top_idx = np.argsort(gene_pcc)[-topk:][::-1]
bot_idx = np.argsort(gene_pcc)[:topk]

print("\n✅ Top PCC genes (Loki 방식):")
for i in top_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

print("\n❌ Bottom PCC genes (Loki 방식):")
for i in bot_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

# =========================
# 8) 결과 저장
# =========================
print("\n" + "="*60)
print("7. 결과 저장")
print("="*60)

np.save("eval_loki_common_sids.npy", np.array(common_sids, dtype=object))
np.save("eval_loki_common_genes.npy", np.array(common_genes, dtype=object))
np.save("eval_loki_gene_pcc.npy", gene_pcc.astype(np.float32))
np.save("eval_loki_gene_mae.npy", gene_mae.astype(np.float32))

print("✅ 저장 완료:")
print("  - eval_loki_common_sids.npy")
print("  - eval_loki_common_genes.npy")
print("  - eval_loki_gene_pcc.npy")
print("  - eval_loki_gene_mae.npy")

print("\n" + "="*60)
print("✅ Loki 방식 평가 완료!")
print("="*60)


1. TCGA RNA 데이터 로딩
RNA slides: 461, RNA genes: 24778

2. PredEx gene list 로딩
PredEx genes: 12009

3. HEG gene list 로딩
HEG genes: 7481

4. 공통 gene 및 slide 매칭
✅ Common genes: 6105
Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']

Pred slides: 331
Common slides: 331

5. 평가 행렬 생성


Building matrices: 100%|██████████| 331/331 [00:00<00:00, 622.21it/s]


Y_true: (331, 6105), Y_pred: (331, 6105)

6. 평가 메트릭 계산
📌 Overall PCC (flatten): 0.6820
📌 Gene-wise PCC mean: 0.0087
📌 Gene-wise MAE mean: 1.4891

✅ Top PCC genes (Loki 방식):
  HTR7: PCC=0.2439, MAE=1.2388
  ELK3: PCC=0.2404, MAE=2.6457
  SLC29A2: PCC=0.2273, MAE=1.4074
  WDR62: PCC=0.2259, MAE=1.0793
  MLLT6: PCC=0.2247, MAE=1.7959
  TNNT1: PCC=0.2242, MAE=2.8741
  PIAS4: PCC=0.2205, MAE=2.1766
  GJA1: PCC=0.2203, MAE=3.8462
  PRRG1: PCC=0.2200, MAE=0.9262
  DHRS7: PCC=0.2166, MAE=2.5071

❌ Bottom PCC genes (Loki 방식):
  ANXA5: PCC=-0.2716, MAE=3.7854
  TSSK6: PCC=-0.2445, MAE=0.6534
  PLCB3: PCC=-0.2437, MAE=2.9967
  MMP10: PCC=-0.2401, MAE=2.9608
  SIRPA: PCC=-0.2395, MAE=2.7196
  ITGA3: PCC=-0.2355, MAE=3.0440
  SERPINE1: PCC=-0.2304, MAE=3.5151
  CYP26B1: PCC=-0.2236, MAE=1.5360
  MMP1: PCC=-0.2180, MAE=3.6101
  AJAP1: PCC=-0.2176, MAE=0.5246

7. 결과 저장
✅ 저장 완료:
  - eval_loki_common_sids.npy
  - eval_loki_common_genes.npy
  - eval_loki_gene_pcc.npy
  - eval_loki_gene_mae.npy

✅ Loki 방

In [14]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

METHOD_NAME = "baseline_idk"  
# METHOD_NAME = "topk_relu_top300"
# METHOD_NAME = "linear"

np.save("Y_true.npy", Y_true.astype(np.float32))
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred.astype(np.float32))

# 이미 gene / slide 정보는 저장돼 있으므로 그대로 사용
np.save("gene_names.npy", np.array(common_genes, dtype=object))
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))

print("\n✅ Saved files:")
print(f" - Y_true.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names.npy: {len(common_genes)} genes")
print(f" - eval_common_sids.npy: {len(common_sids)} slides")



✅ Saved files:
 - Y_true.npy: (331, 6105)
 - Y_pred_baseline_idk.npy: (331, 6105)
 - gene_names.npy: 6105 genes
 - eval_common_sids.npy: 331 slides


## 개선된 Loki 방식 예측 (Top-K + ReLU 가중치)

기존 Loki 방식의 문제점을 개선:
1. **Top-K 필터링**: 가장 유사한 K개만 선택 (노이즈 제거)
2. **ReLU 가중치**: 음수 similarity 제거 (안정성 향상)
3. **Temperature scaling**: 가중치 분포 조정

이 방식은 기존 Cell 14 방식과 유사하지만, Loki의 `predict_st_gene_expr` 함수 구조를 유지합니다.


In [15]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import sys

# Loki 모듈 import
loki_path = "/data/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_pred_expr_loki_improved")  # 개선된 Loki 방식 결과
OUT_DIR.mkdir(exist_ok=True)

EPS = 1e-8
TOPK = 128           # Top-K 선택 (기존 Cell 14와 동일)
TEMPERATURE = 1.0     # Temperature scaling (1.0 = no scaling)

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

print(f"✅ 데이터 로드 완료: {Ntrain}개 학습 샘플, {G}개 유전자")
print(f"설정: TOPK={TOPK}, Temperature={TEMPERATURE}")

# =========================
# 개선된 예측 함수
# =========================
def predict_improved(slide_emb, train_text_emb, train_expr, topk=TOPK, temperature=TEMPERATURE):
    """
    개선된 Loki 방식: Top-K + ReLU 가중치 + Temperature scaling
    
    Args:
        slide_emb: (768,) 슬라이드 임베딩
        train_text_emb: (Ntrain, 768) 학습 텍스트 임베딩
        train_expr: (Ntrain, G) 학습 발현 데이터
        topk: Top-K 선택 개수
        temperature: Temperature scaling
    
    Returns:
        pred: (G,) 예측 발현 벡터
    """
    # 1. Similarity 계산
    similarity = slide_emb @ train_text_emb.T  # (Ntrain,)
    
    # 2. Top-K 선택
    if topk < len(similarity):
        topk_idx = np.argpartition(similarity, -topk)[-topk:]  # Top-K 인덱스
        topk_similarity = similarity[topk_idx]  # (topk,)
        topk_expr = train_expr[topk_idx]  # (topk, G)
    else:
        topk_similarity = similarity
        topk_expr = train_expr
    
    # 3. ReLU + Temperature scaling
    topk_similarity = topk_similarity / temperature
    weights = np.maximum(topk_similarity, 0.0)  # ReLU
    weights = weights / (weights.sum() + EPS)  # 정규화
    
    # 4. 가중 평균
    pred = weights @ topk_expr  # (G,)
    
    return pred.astype(np.float32)

# =========================
# Slide loop
# =========================
slide_files = sorted(EMB_DIR.glob("*.npy"))
print(f"\n총 {len(slide_files)}개 슬라이드 처리 시작...")

for p in tqdm(slide_files):
    slide_id = p.stem
    out_path = OUT_DIR / f"{slide_id}.npy"
    
    # Resume 기능
    FORCE_RECOMPUTE = False
    if not FORCE_RECOMPUTE and out_path.exists():
        continue
    
    # 패치 임베딩 로드 및 정규화
    patch_emb = np.load(p)  # (num_patches, 768)
    patch_emb = patch_emb / (np.linalg.norm(patch_emb, axis=1, keepdims=True) + EPS)
    
    # 슬라이드 임베딩: 모든 패치의 평균
    slide_emb = patch_emb.mean(axis=0)  # (768,)
    slide_emb = slide_emb / (np.linalg.norm(slide_emb) + EPS)
    
    # 개선된 예측
    pred = predict_improved(slide_emb, train_text_emb, train_expr, topk=TOPK, temperature=TEMPERATURE)
    
    np.save(out_path, pred)

print(f"\n✅ 개선된 Loki 방식 예측 완료: {OUT_DIR}")


✅ Loki 모듈 로드 성공
Train text: (154763, 768)
Train expr: (154763, 12009)
✅ 데이터 로드 완료: 154763개 학습 샘플, 12009개 유전자
설정: TOPK=128, Temperature=1.0

총 331개 슬라이드 처리 시작...


100%|██████████| 331/331 [00:17<00:00, 18.68it/s]


✅ 개선된 Loki 방식 예측 완료: tcga_pred_expr_loki_improved


## 개선된 Loki 방식 결과 평가

Cell 21에서 생성한 개선된 예측 결과를 평가합니다.


In [22]:
# Cell 19의 평가 코드를 복사하고 PRED_DIR만 변경
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =========================
# 설정 (Cell 19와 동일하지만 PRED_DIR만 변경)
# =========================
PRED_DIR = Path("tcga_pred_expr_loki_improved")  # 개선된 Loki 방식 결과
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"
HEG_GENE_LIST = "/project_antwerp/hbae/data/high_expression_genes_all.txt"

def norm_sid(x):
    return str(x).split(".")[0]

def norm_gene(g):
    g = str(g).split(".")[0].strip().upper()
    return g

# =========================
# 1-4) 데이터 로딩 및 매칭 (Cell 19와 동일)
# =========================
print("="*60)
print("1. TCGA RNA 데이터 로딩")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()
rna_gene_names = np.array([c.replace("rna_", "") for c in rna_expr_df.columns], dtype=object)
rna_gene_norm = np.array([norm_gene(g) for g in rna_gene_names], dtype=object)

rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}
print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")

print("\n" + "="*60)
print("2. PredEx gene list 로딩")
print("="*60)

pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_gene_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

pred_gene_map = {}
for i, g in enumerate(pred_gene_norm):
    if g not in pred_gene_map:
        pred_gene_map[g] = i

print(f"PredEx genes: {len(pred_gene_norm)}")

print("\n" + "="*60)
print("3. HEG gene list 로딩")
print("="*60)

with open(HEG_GENE_LIST, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]
heg_genes = [norm_gene(g) for g in heg_genes]

print(f"HEG genes: {len(heg_genes)}")

print("\n" + "="*60)
print("4. 공통 gene 및 slide 매칭")
print("="*60)

common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print(f"✅ Common genes: {len(common_genes)}")

rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)

pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids = [p.stem.split(".")[0] for p in pred_files]
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))
print(f"\nPred slides: {len(pred_sid_set)}")
print(f"Common slides: {len(common_sids)}")

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0입니다.")

# =========================
# 5) Y_true, Y_pred 만들기
# =========================
print("\n" + "="*60)
print("5. 평가 행렬 생성")
print("="*60)

N = len(common_sids)
G = len(common_genes)

Y_true = np.zeros((N, G), dtype=np.float32)
Y_pred = np.zeros((N, G), dtype=np.float32)

pred_path_map = {p.stem.split(".")[0]: p for p in pred_files}

for j, sid in enumerate(tqdm(common_sids, desc="Building matrices")):
    ridx = rna_slide_map[sid]
    rna_row = rna_expr_df.iloc[ridx].values.astype(np.float32)
    Y_true[j] = rna_row[rna_idx]

    pred_vec = np.load(pred_path_map[sid]).astype(np.float32)
    if pred_vec.ndim != 1:
        raise ValueError(f"{sid}: pred 벡터가 1D가 아님: {pred_vec.shape}")
    Y_pred[j] = pred_vec[pred_idx]

print(f"Y_true: {Y_true.shape}, Y_pred: {Y_pred.shape}")

# =========================
# 6-7) 평가 메트릭 계산
# =========================
print("\n" + "="*60)
print("6. 평가 메트릭 계산 (개선된 Loki 방식)")
print("="*60)

x = Y_true.reshape(-1).astype(np.float64)
y = Y_pred.reshape(-1).astype(np.float64)

x = x - x.mean()
y = y - y.mean()
overall_pcc = (x @ y) / (np.sqrt((x @ x) * (y @ y)) + 1e-12)
print(f"📌 Overall PCC (flatten): {overall_pcc:.4f}")

Xt = Y_true.astype(np.float64)
Yp = Y_pred.astype(np.float64)

Xt_mean = Xt.mean(axis=0, keepdims=True)
Yp_mean = Yp.mean(axis=0, keepdims=True)

Xt_c = Xt - Xt_mean
Yp_c = Yp - Yp_mean

cov = (Xt_c * Yp_c).sum(axis=0)
stdx = np.sqrt((Xt_c * Xt_c).sum(axis=0))
stdy = np.sqrt((Yp_c * Yp_c).sum(axis=0))

gene_pcc = cov / (stdx * stdy + 1e-12)
gene_pcc[np.isnan(gene_pcc)] = 0.0

gene_mae = np.mean(np.abs(Y_true - Y_pred), axis=0)

print(f"📌 Gene-wise PCC mean: {gene_pcc.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_mae.mean():.4f}")

topk = 20
top_idx = np.argsort(gene_pcc)[-topk:][::-1]
bot_idx = np.argsort(gene_pcc)[:topk]

print("\n✅ Top PCC genes (개선된 Loki 방식):")
for i in top_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

print("\n❌ Bottom PCC genes (개선된 Loki 방식):")
for i in bot_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

# =========================
# 8) 결과 저장
# =========================
print("\n" + "="*60)
print("7. 결과 저장")
print("="*60)

np.save("eval_loki_improved_common_sids.npy", np.array(common_sids, dtype=object))
np.save("eval_loki_improved_common_genes.npy", np.array(common_genes, dtype=object))
np.save("eval_loki_improved_gene_pcc.npy", gene_pcc.astype(np.float32))
np.save("eval_loki_improved_gene_mae.npy", gene_mae.astype(np.float32))

print("✅ 저장 완료")
print("\n" + "="*60)
print("✅ 개선된 Loki 방식 평가 완료!")
print("="*60)


1. TCGA RNA 데이터 로딩
RNA slides: 461, RNA genes: 24778

2. PredEx gene list 로딩
PredEx genes: 12009

3. HEG gene list 로딩
HEG genes: 7481

4. 공통 gene 및 slide 매칭
✅ Common genes: 6105

Pred slides: 331
Common slides: 331

5. 평가 행렬 생성


Building matrices: 100%|██████████| 331/331 [00:00<00:00, 516.55it/s]


Y_true: (331, 6105), Y_pred: (331, 6105)

6. 평가 메트릭 계산 (개선된 Loki 방식)
📌 Overall PCC (flatten): 0.6688
📌 Gene-wise PCC mean: -0.0005
📌 Gene-wise MAE mean: 1.4267

✅ Top PCC genes (개선된 Loki 방식):
  HTR7: PCC=0.2730, MAE=1.2338
  MMP3: PCC=0.2663, MAE=2.8638
  ACTN1: PCC=0.2663, MAE=2.6596
  SHISAL1: PCC=0.2516, MAE=0.9350
  P4HA2: PCC=0.2287, MAE=1.8909
  CD44: PCC=0.2269, MAE=2.8164
  DUSP6: PCC=0.2196, MAE=2.7335
  PI15: PCC=0.2169, MAE=0.5161
  SLC9A1: PCC=0.2157, MAE=2.7445
  LTBP1: PCC=0.2129, MAE=2.6739

❌ Bottom PCC genes (개선된 Loki 방식):
  GCHFR: PCC=-0.2544, MAE=0.1534
  P2RY11: PCC=-0.2388, MAE=0.6592
  CFAP298: PCC=-0.2363, MAE=0.3514
  GALNT12: PCC=-0.2315, MAE=0.9983
  NR2F6: PCC=-0.2303, MAE=2.3455
  AC073111: PCC=-0.2242, MAE=0.4196
  SEMA3A: PCC=-0.2157, MAE=0.6092
  C8G: PCC=-0.2110, MAE=0.5130
  DOT1L: PCC=-0.2072, MAE=1.3001
  BICDL1: PCC=-0.2010, MAE=1.4680

7. 결과 저장
✅ 저장 완료

✅ 개선된 Loki 방식 평가 완료!


In [23]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

METHOD_NAME = "baseline_idk2"  
# METHOD_NAME = "topk_relu_top300"
# METHOD_NAME = "linear"

np.save("Y_true.npy", Y_true.astype(np.float32))
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred.astype(np.float32))

# 이미 gene / slide 정보는 저장돼 있으므로 그대로 사용
np.save("gene_names.npy", np.array(common_genes, dtype=object))
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))

print("\n✅ Saved files:")
print(f" - Y_true.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names.npy: {len(common_genes)} genes")
print(f" - eval_common_sids.npy: {len(common_sids)} slides")



✅ Saved files:
 - Y_true.npy: (331, 6105)
 - Y_pred_baseline_idk2.npy: (331, 6105)
 - gene_names.npy: 6105 genes
 - eval_common_sids.npy: 331 slides


## 📊 기본 Loki 방식 vs 개선된 Loki 방식 비교

### 🔍 핵심 차이점 요약

| 항목 | **기본 Loki 방식** (Cell 15, 17) | **개선된 Loki 방식** (Cell 21) |
|------|--------------------------------|------------------------------|
| **사용 함수** | `loki.predex.predict_st_gene_expr()` | 커스텀 `predict_improved()` |
| **Similarity 사용** | ✅ **모든 similarity** (154,763개 전부) | ✅ **Top-K만** (128개만 선택) |
| **가중치 계산** | 원본 similarity 그대로 사용 | ReLU 적용 (음수 제거) |
| **Temperature** | ❌ 없음 | ✅ Temperature scaling 지원 |
| **노이즈 필터링** | ❌ 없음 (모든 값 사용) | ✅ Top-K로 노이즈 제거 |
| **음수 처리** | ❌ 음수 similarity 포함 | ✅ ReLU로 음수 제거 |

---

### 📝 상세 알고리즘 비교

#### **1. 기본 Loki 방식** (Cell 17)

```python
# 1. Similarity 계산
similarity = slide_emb @ train_text_emb.T  # (154763,) - 모든 학습 샘플과 비교

# 2. Loki 함수 호출 (모든 similarity 사용)
pred = loki.predex.predict_st_gene_expr(similarity, train_expr)
```

**내부 동작 (`loki.predex.predict_st_gene_expr`):**
```python
# 1. 가중 합계 계산
weighted_sum = similarity @ train_expr  # (154763,) @ (154763, G) = (G,)

# 2. 정규화 (모든 similarity의 합으로 나눔)
weights = similarity.sum() + 1e-8  # 모든 similarity 합 (음수 포함!)
pred = weighted_sum / weights
```

**특징:**
- ✅ **장점**: 간단하고 빠름, Loki 공식 방식
- ❌ **단점**: 
  - 모든 similarity 사용 → 노이즈 포함 가능
  - 음수 similarity도 가중치에 포함 → 부적절한 샘플 영향
  - 유사하지 않은 샘플도 예측에 기여

---

#### **2. 개선된 Loki 방식** (Cell 21)

```python
# 1. Similarity 계산
similarity = slide_emb @ train_text_emb.T  # (154763,)

# 2. Top-K 선택 (가장 유사한 128개만)
topk_idx = np.argpartition(similarity, -128)[-128:]
topk_similarity = similarity[topk_idx]  # (128,)
topk_expr = train_expr[topk_idx]  # (128, G)

# 3. ReLU + Temperature scaling
topk_similarity = topk_similarity / temperature  # Temperature 조정
weights = np.maximum(topk_similarity, 0.0)  # ReLU: 음수 → 0
weights = weights / weights.sum()  # 정규화

# 4. 가중 평균
pred = weights @ topk_expr  # (G,)
```

**특징:**
- ✅ **장점**: 
  - Top-K로 노이즈 제거
  - ReLU로 음수 similarity 제거 → 안정성 향상
  - Temperature로 가중치 분포 조정 가능
- ❌ **단점**: 
  - Top-K 선택에 시간 소요 (하지만 미미함)
  - 하이퍼파라미터 조정 필요 (TOPK, TEMPERATURE)

---

### 🧮 수식 비교

#### **기본 Loki 방식:**
```
pred = (Σᵢ similarityᵢ × exprᵢ) / (Σᵢ similarityᵢ)
     = (similarity @ train_expr) / similarity.sum()
```
- 모든 i (1 ~ 154,763)에 대해 계산
- similarityᵢ가 음수여도 포함됨

#### **개선된 Loki 방식:**
```
1. Top-K 선택: similarity_topk = top_k(similarity, k=128)
2. ReLU 적용: weights = max(0, similarity_topk / temperature)
3. 정규화: weights = weights / sum(weights)
4. 예측: pred = weights @ expr_topk
```
- Top-K (128개)만 사용
- 음수 similarity는 0으로 처리

---

### 📈 예상 성능 차이

| 메트릭 | 기본 Loki | 개선된 Loki | 기대 효과 |
|--------|-----------|-------------|----------|
| **Overall PCC** | 0.6820 | ~0.68-0.70 | 유사하거나 약간 향상 |
| **Gene-wise PCC mean** | 0.0087 | **0.01-0.05** | **개선 예상** ⬆️ |
| **Top genes PCC** | ~0.24 | **~0.25-0.30** | **개선 예상** ⬆️ |
| **안정성** | 보통 | **높음** | ReLU로 향상 ⬆️ |

---

### 🎯 언제 어떤 방식을 사용할까?

**기본 Loki 방식 (Cell 17) 사용 시기:**
- ✅ 빠른 프로토타이핑
- ✅ Loki 공식 방식과 동일한 결과 필요
- ✅ 모든 학습 샘플의 정보를 활용하고 싶을 때

**개선된 Loki 방식 (Cell 21) 사용 시기:**
- ✅ 더 높은 성능 필요
- ✅ Gene-wise PCC 개선 필요
- ✅ 노이즈가 많은 데이터셋
- ✅ 하이퍼파라미터 튜닝 가능

---

### 🔧 하이퍼파라미터 튜닝 가이드 (Cell 21)

```python
TOPK = 128        # 추천 범위: 64, 128, 256, 512
                  # 작을수록: 더 정확하지만 다양성 감소
                  # 클수록: 더 다양하지만 노이즈 증가

TEMPERATURE = 1.0  # 추천 범위: 0.5, 1.0, 2.0
                   # 작을수록: 더 sharp한 가중치 (top만 강조)
                   # 클수록: 더 smooth한 가중치 (균등 분포)
```


## 🔍 기존 Cell 14 vs 제가 추가한 Cell 21 비교

### ⚠️ **핵심 차이: 예측 단위가 다릅니다!**

| 항목 | **기존 Cell 14** (패치 레벨) | **제가 추가한 Cell 21** (슬라이드 레벨) |
|------|---------------------------|--------------------------------------|
| **예측 단위** | 각 패치마다 예측 | 슬라이드 전체를 하나로 예측 |
| **입력** | `img_chunk: (P, 768)` - 여러 패치 | `slide_emb: (768,)` - 슬라이드 하나 |
| **출력 형태** | `(num_patches, G)` | `(G,)` |
| **결과 저장** | `tcga_predicted_expression/` | `tcga_pred_expr_loki_improved/` |
| **용도** | 공간 정보 유지 (어느 패치에서 발현?) | 슬라이드 전체 발현 (전체 평균) |

---

### 📝 상세 비교

#### **1. 기존 Cell 14 (패치 레벨 예측)**

```python
def predict_for_patch_chunk(img_chunk: np.ndarray) -> np.ndarray:
    """
    img_chunk: (P, 768) - 여러 패치의 임베딩
    return: (P, G) - 각 패치마다 유전자 발현 예측
    """
    # 각 패치마다 Top-K 선택 + ReLU 가중치
    # 결과: (P, G) - 패치별 예측
```

**특징:**
- ✅ **공간 정보 유지**: 어떤 패치에서 어떤 유전자가 발현하는지 알 수 있음
- ✅ **세밀한 분석 가능**: 패치별 발현 패턴 분석 가능
- ❌ **메모리 많이 사용**: (num_patches, G) 형태로 저장
- ❌ **평가 복잡**: 패치 레벨 평가는 추가 작업 필요

**예시:**
```
슬라이드에 11,834개 패치가 있으면
→ 결과: (11,834, 12,009) 형태
→ 각 패치마다 12,009개 유전자 발현 예측
```

---

#### **2. 제가 추가한 Cell 21 (슬라이드 레벨 예측)**

```python
def predict_improved(slide_emb, ...):
    """
    slide_emb: (768,) - 슬라이드 전체의 평균 임베딩
    return: (G,) - 슬라이드 전체 유전자 발현 예측
    """
    # 슬라이드 임베딩 = 모든 패치의 평균
    slide_emb = patch_emb.mean(axis=0)  # (768,)
    
    # Top-K + ReLU + Temperature
    # 결과: (G,) - 슬라이드 전체 예측
```

**특징:**
- ✅ **간단한 평가**: 슬라이드 레벨로 바로 평가 가능
- ✅ **메모리 효율**: (G,) 형태로 저장
- ✅ **빠른 처리**: 패치별로 반복하지 않음
- ❌ **공간 정보 손실**: 어느 패치에서 발현하는지 모름

**예시:**
```
슬라이드에 11,834개 패치가 있어도
→ 결과: (12,009,) 형태
→ 슬라이드 전체의 평균 유전자 발현 예측
```

---

### 🎯 알고리즘 비교

#### **기존 Cell 14:**
```
1. 각 패치 임베딩 로드: (11,834, 768)
2. 각 패치마다:
   - Top-K 선택
   - ReLU 가중치
   - 가중 평균
3. 결과: (11,834, 12,009) - 패치별 예측
```

#### **제가 추가한 Cell 21:**
```
1. 모든 패치 임베딩 로드: (11,834, 768)
2. 슬라이드 임베딩 생성: 평균 → (768,)
3. 슬라이드 레벨로:
   - Top-K 선택
   - ReLU 가중치
   - Temperature scaling
   - 가중 평균
4. 결과: (12,009,) - 슬라이드 전체 예측
```

---

### 📊 사용 목적에 따른 선택

**기존 Cell 14 (패치 레벨) 사용 시기:**
- ✅ 공간적 발현 패턴 분석 필요
- ✅ 패치별 유전자 발현 비교
- ✅ 공간 전사체학(ST) 데이터와 비교
- ✅ 조직 내 이질성(heterogeneity) 분석

**제가 추가한 Cell 21 (슬라이드 레벨) 사용 시기:**
- ✅ Bulk RNA-seq와 비교 평가
- ✅ 슬라이드 간 비교 분석
- ✅ 빠른 프로토타이핑
- ✅ 메모리 효율 필요

---

### 💡 결론

**다릅니다!** 하지만 알고리즘은 거의 동일합니다:

| 알고리즘 요소 | 기존 Cell 14 | 제가 추가한 Cell 21 |
|-------------|------------|-------------------|
| **Top-K 선택** | ✅ 있음 (128) | ✅ 있음 (128) |
| **ReLU 가중치** | ✅ 있음 | ✅ 있음 |
| **Temperature** | ❌ 없음 | ✅ 있음 (추가) |
| **예측 단위** | 패치별 | 슬라이드 전체 |

**차이점:**
1. **예측 단위**: 패치 vs 슬라이드
2. **Temperature scaling**: Cell 21에만 있음
3. **용도**: 공간 분석 vs 전체 평가


## 🔬 기본 Loki 방식 예측 메커니즘 상세 설명

### 📋 전체 프로세스 개요

기본 Loki 방식은 **Vision-Language 모델의 공통 임베딩 공간**을 활용하여 예측합니다.

```
이미지 임베딩 (768차원) ←→ 텍스트 임베딩 (768차원)
        ↓                           ↓
    Cosine Similarity 계산
        ↓
    가중 평균으로 유전자 발현 예측
```

---

### 🎯 Cell 17 (슬라이드 레벨) 예측 메커니즘

#### **Step 1: 슬라이드 임베딩 생성**

```python
# 1-1. 패치 임베딩 로드
patch_emb = np.load("tcga_embeddings/TCGA-XXX.npy")  # (11,834, 768)
# 예: 슬라이드에 11,834개 패치가 있음

# 1-2. 각 패치 정규화 (L2 normalize)
patch_emb = patch_emb / ||patch_emb||  # (11,834, 768)

# 1-3. 슬라이드 임베딩 = 모든 패치의 평균
slide_emb = patch_emb.mean(axis=0)  # (768,)
# → 11,834개 패치를 평균내어 하나의 벡터로 만듦

# 1-4. 슬라이드 임베딩 정규화
slide_emb = slide_emb / ||slide_emb||  # (768,)
```

**의미:**
- 각 패치는 768차원 벡터
- 슬라이드 전체를 하나의 768차원 벡터로 표현
- 공간 정보는 손실되지만, 슬라이드 전체 특성을 요약

---

#### **Step 2: Similarity 계산 (모든 학습 샘플과 비교)**

```python
# 2-1. 학습 텍스트 임베딩 (이미 정규화됨)
train_text_emb  # (154,763, 768)
# → 154,763개 학습 샘플의 텍스트 임베딩

# 2-2. Cosine Similarity 계산 (내적 = 코사인 유사도)
similarity = slide_emb @ train_text_emb.T  # (154,763,)
# → 슬라이드와 각 학습 샘플 간의 유사도

# 예시 결과:
# similarity[0] = 0.85  → 첫 번째 학습 샘플과 매우 유사
# similarity[1] = 0.12  → 두 번째 학습 샘플과 덜 유사
# similarity[2] = -0.23 → 세 번째 학습 샘플과 반대 방향 (음수!)
# ...
# similarity[154762] = 0.05  → 마지막 학습 샘플과 거의 무관
```

**특징:**
- ✅ **모든 학습 샘플과 비교** (154,763개 전부)
- ✅ **음수 similarity 포함** (코사인 유사도는 -1 ~ 1 범위)
- ❌ **Top-K 선택 없음** (유사하지 않은 샘플도 포함)

---

#### **Step 3: Loki PredEx 함수 호출**

```python
# 3-1. 입력 준비
similarity = similarity.reshape(1, -1)  # (1, 154763)
# → Loki 함수는 2D 배열을 기대하므로 reshape

train_expr  # (154,763, 12,009)
# → 각 학습 샘플의 실제 유전자 발현 값

# 3-2. Loki 함수 호출
pred = loki.predex.predict_st_gene_expr(similarity, train_expr)
# → (1, 12,009) 형태로 반환
pred = pred[0]  # (12,009,) - 1D로 변환
```

---

#### **Step 4: Loki 함수 내부 동작 (핵심!)**

```python
# loki.predex.predict_st_gene_expr 내부:

# 4-1. 가중 합계 계산
weighted_sum = similarity @ train_expr
# (1, 154763) @ (154763, 12009) = (1, 12009)
# 
# 수식으로 표현하면:
# weighted_sum[gene_j] = Σᵢ (similarity[i] × train_expr[i, gene_j])
#                      = similarity[0]×expr[0,j] + similarity[1]×expr[1,j] + ... + similarity[154762]×expr[154762,j]

# 4-2. 정규화 인자 계산 (모든 similarity의 합)
weights = similarity.sum(axis=1, keepdims=True) + 1e-8
# (1, 1)
# = similarity[0] + similarity[1] + ... + similarity[154762] + 1e-8
# 
# ⚠️ 주의: 음수 similarity도 합에 포함됨!
# 예: 0.85 + 0.12 + (-0.23) + ... + 0.05 = 전체 합

# 4-3. 정규화 (가중 평균)
predicted = weighted_sum / weights
# (1, 12009) / (1, 1) = (1, 12009)
```

**수식으로 표현:**

```
pred[gene_j] = (Σᵢ similarity[i] × train_expr[i, gene_j]) / (Σᵢ similarity[i])
```

**의미:**
- 각 유전자마다, 모든 학습 샘플의 발현값을 similarity로 가중 평균
- 가중치 = similarity 값 (음수 포함!)
- 정규화 = 모든 similarity의 합으로 나눔

---

### 🔍 구체적인 예시

**가정:**
- 슬라이드 임베딩: `slide_emb = (768,)`
- 학습 샘플 3개만 있다고 가정 (실제로는 154,763개)

**Step 1: Similarity 계산**
```
similarity[0] = 0.85  (매우 유사)
similarity[1] = 0.12  (덜 유사)
similarity[2] = -0.23 (반대 방향, 음수!)
```

**Step 2: 유전자 A 발현 예측**
```
train_expr[0, gene_A] = 5.2  (첫 번째 샘플의 유전자 A 발현)
train_expr[1, gene_A] = 2.1  (두 번째 샘플의 유전자 A 발현)
train_expr[2, gene_A] = 8.7  (세 번째 샘플의 유전자 A 발현)

weighted_sum = 0.85×5.2 + 0.12×2.1 + (-0.23)×8.7
             = 4.42 + 0.252 - 2.001
             = 2.671

weights = 0.85 + 0.12 + (-0.23) = 0.74

pred[gene_A] = 2.671 / 0.74 = 3.61
```

**문제점:**
- 음수 similarity(-0.23)가 포함되어 예측에 부정적 영향을 줌
- 유사하지 않은 샘플도 예측에 기여

---

### 📊 Cell 15 (패치 레벨) 예측 메커니즘

Cell 15는 **각 패치마다** 동일한 과정을 반복합니다:

```python
# 각 패치마다:
for patch_emb in patch_embeddings:  # (768,)
    # 1. Similarity 계산
    similarity = patch_emb @ train_text_emb.T  # (154763,)
    
    # 2. Loki 함수 호출
    pred_patch = loki.predex.predict_st_gene_expr(similarity, train_expr)
    # → (12,009,) - 이 패치의 유전자 발현 예측
```

**결과:**
- 슬라이드에 11,834개 패치가 있으면
- 11,834개의 예측 벡터 생성: `(11,834, 12,009)`
- 각 패치마다 독립적으로 예측

---

### ⚠️ 기본 Loki 방식의 특징

**장점:**
- ✅ 간단하고 빠름
- ✅ Loki 공식 방식 (표준)
- ✅ 모든 학습 샘플의 정보 활용

**단점:**
- ❌ **음수 similarity 포함** → 부적절한 샘플이 예측에 부정적 영향
- ❌ **노이즈 포함** → 유사하지 않은 샘플도 예측에 기여
- ❌ **정규화 문제** → similarity 합이 0에 가까우면 불안정

**예시 문제:**
```
similarity = [0.1, 0.05, -0.08, -0.07, ...]
sum = 0.1 + 0.05 + (-0.08) + (-0.07) + ... ≈ 0
→ 정규화 시 불안정하거나 의미 없는 예측
```

---

### 💡 핵심 요약

**기본 Loki 방식의 예측 공식:**

```
pred[gene_j] = (Σᵢ similarity[i] × train_expr[i, gene_j]) / (Σᵢ similarity[i])
```

**특징:**
1. **모든 similarity 사용** (154,763개 전부)
2. **음수 similarity 포함** (ReLU 없음)
3. **Top-K 선택 없음** (필터링 없음)
4. **가중 평균** (similarity가 가중치)

이것이 기본 Loki 방식의 핵심 메커니즘입니다!


## Gene 목록 개수 확인

ref_file, RNA 데이터, 예측 결과, 공통 gene의 개수를 확인합니다.


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 경로 설정
# =========================
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"
HEG_GENE_LIST = "/project_antwerp/hbae/data/high_expression_genes_all.txt"

def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

# =========================
# 1. ref_file에서 읽은 gene 목록
# =========================
rna_df = pd.read_csv(TCGA_RNA_CSV)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

# 숫자 컬럼만 선택 (이것이 실제 gene expression 컬럼)
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# rna_ prefix 제거
rna_gene_names = [c.replace("rna_", "") for c in rna_expr_df.columns]

ref_genes = rna_gene_names  # ref_file에서 읽은 gene 목록

# =========================
# 2. RNA 데이터 컬럼 수
# =========================
rna_df_genes = rna_expr_df.shape[1]  # TCGA expression matrix 컬럼 수

# =========================
# 3. 예측 결과 gene 수 (all_shared_genes.txt)
# =========================
pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_df_genes = len(pred_genes_raw)  # 예측 결과 컬럼 수

# =========================
# 4. 공통 gene 수 (최종 비교에 사용된 gene)
# =========================
# HEG gene list 로딩
with open(HEG_GENE_LIST, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]
heg_genes = [norm_gene(g) for g in heg_genes]

# 정규화
rna_gene_norm = [norm_gene(g) for g in rna_gene_names]
pred_gene_norm = [norm_gene(g) for g in pred_genes_raw]

# 공통 gene 계산
rna_gene_set = set(rna_gene_norm)
pred_gene_set = set(pred_gene_norm)
heg_gene_set = set(heg_genes)

common_genes = sorted(rna_gene_set & pred_gene_set & heg_gene_set)
shared_genes = len(common_genes)  # 최종 비교 gene 수

# =========================
# 출력
# =========================
print("="*60)
print("Gene 목록 개수 확인")
print("="*60)
print(f"ref_file genes: {len(ref_genes)}")          # ref_file에서 읽은 gene 목록
print(f"rna_df genes: {rna_df_genes}")              # TCGA expression matrix 컬럼 수
print(f"pred_df genes: {pred_df_genes}")            # 예측 결과 컬럼 수
print(f"shared_genes: {shared_genes}")               # 최종 비교 gene 수 (=12009)
print("="*60)


Gene 목록 개수 확인
ref_file genes: 24778
rna_df genes: 24778
pred_df genes: 12009
shared_genes: 6105


In [25]:
import numpy as np
from pathlib import Path

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression")
OUT_DIR.mkdir(exist_ok=True)

TOPK = 128           # 64~256 추천 (클수록 정확↑, 메모리/시간↑)
PATCH_CHUNK = 256    # 128~1024 (RAM/속도에 맞춰)
TRAIN_CHUNK = 20000  # 10000~50000 (RAM에 맞춰)
EPS = 1e-8

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize (cosine = dot product)
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

def predict_for_patch_chunk(img_chunk: np.ndarray) -> np.ndarray:
    """
    img_chunk: (P, 768) float32, L2-normalized
    return: (P, G)
    """
    P = img_chunk.shape[0]

    top_scores = np.full((P, TOPK), -np.inf, dtype=np.float32)
    top_indices = np.full((P, TOPK), -1, dtype=np.int32)

    # train chunk-wise similarity
    for t0 in range(0, Ntrain, TRAIN_CHUNK):
        t1 = min(t0 + TRAIN_CHUNK, Ntrain)
        tr = train_text_emb[t0:t1]               # (Tc, 768)
        S = img_chunk @ tr.T                     # (P, Tc)

        k = min(TOPK, S.shape[1])
        idx_part = np.argpartition(S, -k, axis=1)[:, -k:]          # (P, k)
        sc_part  = np.take_along_axis(S, idx_part, axis=1)         # (P, k)
        idx_part = idx_part + t0                                   # global idx

        cand_scores = np.concatenate([top_scores, sc_part], axis=1)
        cand_idx    = np.concatenate([top_indices, idx_part], axis=1)

        sel = np.argpartition(cand_scores, -TOPK, axis=1)[:, -TOPK:]
        top_scores  = np.take_along_axis(cand_scores, sel, axis=1)
        top_indices = np.take_along_axis(cand_idx, sel, axis=1)

    # ✅ 안정적인 weight: ReLU(sim) 후 정규화 (음수 cosine 때문에 sum이 0되는 문제 방지)
    w = np.maximum(top_scores, 0.0)
    w = w / (w.sum(axis=1, keepdims=True) + EPS)   # (P, TOPK)

    # gather expr: (P, TOPK, G) then weighted sum -> (P, G)
    gathered = train_expr[top_indices]             # (P, TOPK, G)
    pred = (w[:, :, None] * gathered).sum(axis=1).astype(np.float32)
    return pred

# =========================
# Slide loop
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    # resume
    if out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id}")

    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        pred_chunks.append(predict_for_patch_chunk(img_emb[p0:p1]))

    pred_expr = np.vstack(pred_chunks)
    out_path = OUT_DIR / f"{slide_id}.npy"
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"

    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape}")


print("✅ Done.")


Train text: (154763, 768)
Train expr: (154763, 12009)
⏭️ Skip (exists): TCGA-4P-AA8J-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-4074-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-4077-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-5151-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-5153-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-6870-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-7269-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6D8-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DA-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DB-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DD-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DE-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DG-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DI-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DJ-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A6DL-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-A8YP-01Z-00-DX1
⏭️ Skip (exists): TCGA-BB-4217-01Z-00-DX1
⏭️ Skip (exists): TCGA-BB-4225-01Z-00-DX1
⏭️ Skip (exists): TCGA-BB-4227-01Z-00-DX1
⏭️ Skip (exists): TCGA-BB-7866-01Z-00-DX1
⏭️ Skip (exists): TCGA-BB-7871-01Z-00-DX1
⏭️ Skip (exists): TCGA

In [26]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# paths
emb_dir = Path("tcga_embeddings")          # slide patch embeddings
out_dir = Path("tcga_pred_expr")          # predicted expression output
out_dir.mkdir(exist_ok=True)

# already loaded:
# train_text_emb: (N_train, 768)
# train_expr    : (N_train, G)

def l2norm(x, axis=-1, eps=1e-8):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def predex_slide(slide_patch_emb, train_text_emb, train_expr, topk=2048, temperature=1.0):
    # 1) pool patches -> slide embedding
    slide_emb = slide_patch_emb.mean(axis=0)  # (768,)
    slide_emb = l2norm(slide_emb)

    # 2) similarity
    sim = (slide_emb @ train_text_emb.T) / temperature  # (N_train,)

    # 3) top-k
    if topk is not None and topk < sim.shape[0]:
        idx = np.argpartition(sim, -topk)[-topk:]
        sel_sim = sim[idx]
        # stable softmax
        sel_sim = sel_sim - sel_sim.max()
        w = np.exp(sel_sim)
        w = w / (w.sum() + 1e-8)             # (topk,)
        pred = w @ train_expr[idx]           # (G,)
    else:
        sim = sim - sim.max()
        w = np.exp(sim)
        w = w / (w.sum() + 1e-8)
        pred = w @ train_expr

    return pred.astype(np.float32)

# run all slides
slide_files = sorted(emb_dir.glob("*.npy"))
print("Slides:", len(slide_files))

for p in tqdm(slide_files):
    slide_id = p.stem
    out_path = out_dir / f"{slide_id}.npy"
    if out_path.exists():
        continue

    patch_emb = np.load(p)                 # (num_patches, 768)
    patch_emb = l2norm(patch_emb)          # normalize patches (good practice)

    pred = predex_slide(patch_emb, train_text_emb, train_expr, topk=2048, temperature=1.0)
    np.save(out_path, pred)

print("✅ Done: TCGA slide-level predicted expression saved to", out_dir)


Slides: 331


100%|██████████| 331/331 [00:00<00:00, 63404.94it/s]

✅ Done: TCGA slide-level predicted expression saved to tcga_pred_expr


In [27]:
list(pred_dict.keys())[:10]
rna_df.index[:10].tolist()


NameError: name 'pred_dict' is not defined

In [28]:
import numpy as np
from pathlib import Path

p = next(Path("tcga_pred_expr").glob("*.npy"))
x = np.load(p)
print(p.name, x.shape)


TCGA-WA-A7GZ-01Z-00-DX2.npy (12009,)


In [29]:
from pathlib import Path

heg_path = Path("/project_antwerp/hbae/data/high_expression_genes_all.txt")

with open(heg_path, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]

print("HEG genes:", len(heg_genes), "example:", heg_genes[:5])


HEG genes: 7481 example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2']


In [30]:
import numpy as np
import pandas as pd

# -----------------------
# 0) RNA slide id 정규화
# -----------------------
RNA_SLIDE_COL = "wsi_file_name"

def norm_sid(x):
    return str(x).split(".")[0]

rna_df = rna_df.copy()
rna_df["sid_norm"] = rna_df[RNA_SLIDE_COL].map(norm_sid)
rna_df = rna_df.drop_duplicates("sid_norm")

rna_slide_map = {sid: i for i, sid in enumerate(rna_df["sid_norm"].values)}

# -----------------------
# 1) RNA expression: 숫자 컬럼만
# -----------------------
rna_expr_df = rna_df.select_dtypes(include=[np.number])

print("RNA expr shape:", rna_expr_df.shape)

# 🔴 핵심: rna_ prefix 제거
rna_genes_raw = np.array(
    [c.replace("rna_", "") for c in rna_expr_df.columns],
    dtype=object
)

print("RNA genes example:", rna_genes_raw[:10])


RNA expr shape: (461, 24778)
RNA genes example: ['TSPAN6' 'DPM1' 'SCYL3' 'C1orf112' 'FGR' 'CFH' 'FUCA2' 'GCLC' 'NFYA'
 'STPG1']


In [31]:
# PredEx에서 사용한 gene list
pred_genes_raw = np.array(
    [g.strip() for g in open("/project_antwerp/hbae/data/all_shared_genes.txt") if g.strip()],
    dtype=object
)

print("Pred genes example:", pred_genes_raw[:10])
print("Pred genes count:", len(pred_genes_raw))


Pred genes example: ['A1CF' 'A2M' 'A2ML1' 'A3GALT2' 'A4GALT' 'A4GNT' 'AACS' 'AADAC' 'AADACL2'
 'AADACL3']
Pred genes count: 12009


In [32]:
def norm_gene(g):
    g = str(g)
    g = g.split(".")[0]      # 혹시 ENSG.xxx.15 같은 거 대비
    g = g.strip().upper()
    return g

rna_norm  = np.array([norm_gene(g) for g in rna_genes_raw], dtype=object)
pred_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

rna_gene_map = {}
for i, g in enumerate(rna_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print("✅ Common HEG genes:", len(common_genes))


print("✅ Common genes:", len(common_genes))
print("Example:", common_genes[:10])


✅ Common HEG genes: 6095
✅ Common genes: 6095
Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']


In [33]:
rna_idx  = np.array([rna_gene_map[g]  for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)


In [34]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# -----------------------
# 설정
# -----------------------
RNA_SLIDE_COL = "wsi_file_name"  # 너가 말한 컬럼명
PRED_DIR = Path("tcga_pred_expr")  # 네 pred slide-level .npy 저장한 폴더로 맞춰줘
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"

def norm_sid(x):
    return str(x).split(".")[0]

def norm_gene(g):
    g = str(g).split(".")[0].strip().upper()
    return g

# -----------------------
# 1) RNA: slide id + numeric expression만
# -----------------------
rna_df2 = rna_df.copy()
rna_df2["sid_norm"] = rna_df2[RNA_SLIDE_COL].map(norm_sid)
rna_df2 = rna_df2.drop_duplicates("sid_norm")

# 숫자 컬럼만 뽑기 (문자열 컬럼 TCGA-HNSC 같은거 제거됨)
rna_expr_df = rna_df2.select_dtypes(include=[np.number]).copy()

# RNA gene 이름: rna_ prefix 제거
rna_gene_names = np.array([c.replace("rna_", "") for c in rna_expr_df.columns], dtype=object)
rna_gene_norm  = np.array([norm_gene(g) for g in rna_gene_names], dtype=object)

# RNA gene map (중복 있으면 첫번째만)
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df2["sid_norm"].values)}

print("RNA slides:", len(rna_slide_map), "RNA genes(numeric):", rna_expr_df.shape[1])

# -----------------------
# 2) Pred: gene list (12009) 로드
# -----------------------
pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_gene_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

pred_gene_map = {}
for i, g in enumerate(pred_gene_norm):
    if g not in pred_gene_map:
        pred_gene_map[g] = i

print("Pred genes:", len(pred_gene_norm))

# -----------------------
# 3) 공통 gene 만들기 (10331 나와야 정상)
# -----------------------
common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print("✅ Common genes:", len(common_genes), "Example:", common_genes[:10])

rna_idx  = np.array([rna_gene_map[g]  for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)

# -----------------------
# 4) 공통 slide 교집합 만들기
# -----------------------
pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids = [p.stem.split(".")[0] for p in pred_files]  # 혹시 .npy.tmp 같은거 대비
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))
print("Pred slides:", len(pred_sid_set))
print("Common slides:", len(common_sids))

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0이야. sid_norm 로직이나 pred 파일 이름 확인 필요")

# -----------------------
# 5) Y_true, Y_pred 만들기 (N_slides, N_common_genes)
# -----------------------
N = len(common_sids)
G = len(common_genes)

Y_true = np.zeros((N, G), dtype=np.float32)
Y_pred = np.zeros((N, G), dtype=np.float32)

# pred 파일 빠르게 찾기 dict
pred_path_map = {p.stem.split(".")[0]: p for p in pred_files}

for j, sid in enumerate(tqdm(common_sids, desc="Building matrices")):
    # RNA
    ridx = rna_slide_map[sid]
    rna_row = rna_expr_df.iloc[ridx].values.astype(np.float32)  # (RNA_numeric_genes,)
    Y_true[j] = rna_row[rna_idx]

    # Pred (12009,)
    pred_vec = np.load(pred_path_map[sid]).astype(np.float32)
    if pred_vec.ndim != 1:
        raise ValueError(f"{sid}: pred 벡터가 1D가 아님: {pred_vec.shape}")
    Y_pred[j] = pred_vec[pred_idx]

print("Y_true:", Y_true.shape, "Y_pred:", Y_pred.shape)

# -----------------------
# 6) Overall PCC (flatten)
# -----------------------
x = Y_true.reshape(-1).astype(np.float64)
y = Y_pred.reshape(-1).astype(np.float64)

x = x - x.mean()
y = y - y.mean()
overall_pcc = (x @ y) / (np.sqrt((x @ x) * (y @ y)) + 1e-12)
print(f"\n📌 Overall PCC (flatten): {overall_pcc:.4f}")

# -----------------------
# 7) Gene-wise PCC (벡터화)
# -----------------------
Xt = Y_true.astype(np.float64)
Yp = Y_pred.astype(np.float64)

Xt_mean = Xt.mean(axis=0, keepdims=True)
Yp_mean = Yp.mean(axis=0, keepdims=True)

Xt_c = Xt - Xt_mean
Yp_c = Yp - Yp_mean

cov = (Xt_c * Yp_c).sum(axis=0)
stdx = np.sqrt((Xt_c * Xt_c).sum(axis=0))
stdy = np.sqrt((Yp_c * Yp_c).sum(axis=0))

gene_pcc = cov / (stdx * stdy + 1e-12)
gene_pcc[np.isnan(gene_pcc)] = 0.0

gene_mae = np.mean(np.abs(Y_true - Y_pred), axis=0)

print(f"📌 Gene-wise PCC mean: {gene_pcc.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_mae.mean():.4f}")

# top/bottom genes
topk = 20
top_idx = np.argsort(gene_pcc)[-topk:][::-1]
bot_idx = np.argsort(gene_pcc)[:topk]

print("\n✅ Top PCC genes:")
for i in top_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

print("\n❌ Bottom PCC genes:")
for i in bot_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

# 필요하면 저장
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))
np.save("eval_common_genes.npy", np.array(common_genes, dtype=object))
np.save("eval_gene_pcc.npy", gene_pcc.astype(np.float32))
np.save("eval_gene_mae.npy", gene_mae.astype(np.float32))
print("\n✅ Saved: eval_common_sids.npy, eval_common_genes.npy, eval_gene_pcc.npy, eval_gene_mae.npy")


RNA slides: 461 RNA genes(numeric): 24778
Pred genes: 12009
✅ Common genes: 6095 Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']
Pred slides: 331
Common slides: 331


Building matrices: 100%|██████████| 331/331 [00:00<00:00, 647.68it/s]

Y_true: (331, 6095) Y_pred: (331, 6095)

📌 Overall PCC (flatten): 0.6879
📌 Gene-wise PCC mean: -0.0019
📌 Gene-wise MAE mean: 1.4451

✅ Top PCC genes:
  PTPN18: PCC=0.2494, MAE=1.8371
  REXO2: PCC=0.2442, MAE=1.7215
  NRBF2: PCC=0.2329, MAE=2.8679
  DOK4: PCC=0.2320, MAE=2.0253
  SLC25A19: PCC=0.2257, MAE=0.9597
  THSD1: PCC=0.2255, MAE=1.8790
  ELK3: PCC=0.2185, MAE=2.6341
  GZMH: PCC=0.2147, MAE=1.3225
  SLC29A2: PCC=0.2147, MAE=1.3768
  ULBP3: PCC=0.2124, MAE=1.5378

❌ Bottom PCC genes:
  PLEC: PCC=-0.2540, MAE=2.9695
  TSSK6: PCC=-0.2329, MAE=0.6460
  PTHLH: PCC=-0.2257, MAE=2.8897
  MGAT3: PCC=-0.2196, MAE=0.5055
  SIRPA: PCC=-0.2191, MAE=2.7117
  TTPAL: PCC=-0.2182, MAE=1.9380
  ATN1: PCC=-0.2118, MAE=2.9990
  FSTL3: PCC=-0.2063, MAE=2.5273
  SH2D5: PCC=-0.2025, MAE=1.2624
  ANXA5: PCC=-0.2007, MAE=3.6250

✅ Saved: eval_common_sids.npy, eval_common_genes.npy, eval_gene_pcc.npy, eval_gene_mae.npy


In [35]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

#METHOD_NAME = "baseline"  
METHOD_NAME = "topk_relu"
# METHOD_NAME = "linear"

np.save("Y_true.npy", Y_true.astype(np.float32))
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred.astype(np.float32))

# 이미 gene / slide 정보는 저장돼 있으므로 그대로 사용
np.save("gene_names.npy", np.array(common_genes, dtype=object))
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))

print("\n✅ Saved files:")
print(f" - Y_true.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names.npy: {len(common_genes)} genes")
print(f" - eval_common_sids.npy: {len(common_sids)} slides")



✅ Saved files:
 - Y_true.npy: (331, 6095)
 - Y_pred_topk_relu.npy: (331, 6095)
 - gene_names.npy: 6095 genes
 - eval_common_sids.npy: 331 slides
